# Perfect stock systeem

In [2]:
import Functions_stock_generation as fsg # Import stock generation functions
import Initialize_product as ip # Call Variables from initialization


from pmf_stock_pred import pmf_stock_predictions_up as pmf_up_script# Call functions from updated pmf prediction script
from Dirichlet_stock_pred import Functions_dirichlet_pred as Dir_script# Call functions from dirichlet prediction script



import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt


#############################################Create data set#############################################

#Set seed for random generations
np.random.seed(19)
#Set seed for stock prediction
rng = np.random.default_rng(42)
            
#Initialize dataframe and time periods
all_products = []
columns = ["Product", "time", "theft", "Price", "Size",
           "stock", "sold", "Rest", "Double scanned", "Extra colli",
           "Colli", "zerotelling"]

empty_row = pd.DataFrame([{col: "" for col in columns}]) # create empty row for readability
T = 365*2  # total time periods 2 years!
Shelf_length = 1 # 1 meters
nr_products = 10 #At 100 (but take 1000 to be sure!) you should be fine with your boxplots! no need to do more. 
Idx_pred = int(np.round(T/2)) #Set Idx for stock prediction 


#initialize variables for short and long timescales pmf updating 
window = 28 #4 weeks, trend (note this can matter a lot)
lag = 50 #time scale for rolling window base_pmf
decay = 0.9 #decay used for updating the pmf
rho = 0.03 #parameter for updating the base_pmf
sigma = 0.8 #parameter for updating the base_pmf


base_prior_pmf = 0 #for updating the base_pmf
states = 0 #for updating the states in base_pmf
history = [] #keep track of pmf history
draws = [] #keep track of single draws
multi_draws = [] #keep track of all draws
base_history = [] #keep track of base_pmf history 
alpha_history = [] ##keep track of Dirichlet history
delivered_stock_sum = 0 #Variable to keep track of deliveries for predictions 


#Calculations for size (needed for theft later on??)
# max_lenght, max_width, max_height = 0.25, 0.25, 0.25
# min_lenght, min_width, min_height = 0.1, 0.1, 0.1
# min_size = min_lenght * min_width * min_height 
# max_size = max_lenght * max_width * max_height 


#Flags

Flag_col_del_mis = False # Missing collies
Flag_col_del_prod_mis = False #Missing products within collies (normal delivery only)

Flag_col_del_extra = False # Not registerd extra colli delivery 
Flag_col_extra_del_prod_mis = False # Missing products within extra collies
Flag_double_scanned = True # Products accidently scanned double or triple

Flagzerotelling = True # Perform a zerotelling
Flagstockcorrecties = True # Perform stock correctie
Flagdayremnants = True # Perform checks in case the colli from the delivery does not fit in the shelf 
Flagorderprediction_pmf = False # Predict when to order new products based on probability mass function 
Flagorderprediction_dir = True # Predict when to order new products based on dirichlet mass function




#################################################Start generation of data###############################################


for product_id in range(0, nr_products): #Loop over alll products
    prod_init = ip.Initialize_prod(T, Shelf_length) 
    price = prod_init["price"]
    day_of_week = prod_init["day_of_week"]
    season = prod_init["season"]   
    theft = prod_init["theft"]
    sold = prod_init["sold"]
    rest = prod_init["rest"]
    max_prod_rows = prod_init["max_prod_rows"]
    size = prod_init["size"]
    prod_max = prod_init["prod_max"]
    prod_min = prod_init["prod_min"]
    Colli = prod_init["Colli"]
    max_delivery = prod_init["max_delivery"]
    Delivery_time = prod_init["Delivery_time"]
    days_after_order = prod_init["days_after_order"]
    Delivery_scheduled = prod_init["Delivery_scheduled"]
    new_order = prod_init["new_order"]
    delivery_amount = prod_init["delivery_amount"]
    double_scanned = prod_init["double_scanned"]
    extra_colli = prod_init["extra_colli"]
    Extra_colli_stock = prod_init["Extra_colli_stock"]
    coll_stat = prod_init["coll_stat"]
    stock_missing = prod_init["stock_missing"]
    zerotell = prod_init["zerotell"]
    zero_t = prod_init["zero_t"]
    stock = prod_init["stock"]
    actual_stock = prod_init["actual_stock"]
    stock_correctie = prod_init["stock_correctie"]
    stock_cor = prod_init["stock_cor"]
    day_rest_cor = prod_init["day_rest_cor"]
    pred_t = prod_init["pred_t"]
    pred_q10 = prod_init["pred_q10"] 
    pred_q50 = prod_init["pred_q50"]  
    pred_q90 = prod_init["pred_q90"]    
    avg_delivery_time = prod_init["avg_delivery_time"] 
    days_till_order = prod_init["days_till_order"]
    Frist_order_day = prod_init["Frist_order_day"]
    time_to_order = prod_init["time_to_order"]
    alpha = prod_init["alpha"]
    alpha_base = prod_init["alpha_base"]
    states = prod_init["states"]
    stock_q10 = prod_init["stock_q10"]
    stock_q50 = prod_init["stock_q50"]
    stock_q90 = prod_init["stock_q90"]
    Exp_date = prod_init["Exp_date"]
        
    print("product_id = ",product_id)



    
    #Will likely change this! put in initialize_product later 
    #prod_min = 10 #CHANGE!!!!!    
    # min_exp_date = np.mean(sold)
    
    # min_exp_date = int(round(min_exp_date*Colli) * 1.5) #1.5 to make a more realistic minimum 
    # #max(1, int(round(min_exp_date * Colli) * 1.5))
    # max_exp_date = int(min_exp_date * 10) # times 10 to try and engulf as many items as possible
    # Exp_date = np.random.randint(min_exp_date, max_exp_date)
    
    
    Initial_pool = [Exp_date] * actual_stock[0]
    pools = [[]]

    counted_negative_pools = set()

    #initialize expired products
    expired_products = np.zeros(T, dtype=int)


    
    
    # ====================================================================================================================                               
    #Loop over every timestep 
    # ====================================================================================================================                               
    for t in range(1, T):

        #Precompute actual stock
        actual_next_stock = (
            actual_stock[t-1]
            - sold[t]
            - theft[t]
            - rest[t]
            #+ Vault_coll_prod # In case colli was missing 1 product!
            )

        if actual_next_stock < 0:
            sold[t], theft[t], rest[t] = fsg.allocate_stock(
                actual_stock[t-1],
                sold[t],
                theft[t],
                rest[t]
            )

            #Correct actual stock
            actual_next_stock = (
                actual_stock[t-1]
                - sold[t]
                - theft[t]
                - rest[t]
                #+ Vault_coll_prod # In case colli was missing 1 product!
                )
        
        #Compute next stock in system with precomputed losses
        next_stock = (
            stock[t-1]
            - sold[t]
            - theft[t]
            - rest[t]
            #+ Vault_coll_prod # In case colli was missing 1 product!
            )
        
        # ====================================================================================================================                               
        #Manual zero-telling (System says stock is present but reality says we have no products)####
        # Check every 7 days, 
        #done in the morning!
        # ====================================================================================================================                               
        if t % 7 == 0 and t <= T and Flagzerotelling == True: 
            #print("zero telling mogelijkheid t =" , t)
            
            if actual_stock[t-1] == 0:
                #print("product_id =", product_id)
                #print("Correction t =",t)
                next_stock = 0
                zero_t[t] = 1
                sold[t] =0 
                theft[t] = 0
                rest[t] = 0
                actual_next_stock = 0
                #print("zero telling t =", t)
        else:
            zero_t[t] = 0
        
################################################################################################################
        # ============================================================
        # 1. Update prediction-based order countdown
        # ============================================================
        
        if not np.isnan(time_to_order):
            # A new predicted order moment was found
            days_till_order = int(time_to_order) - 1
            first_order_day = True
        
        elif days_till_order is not None and days_till_order > 0:
            # Count down toward predicted order day
            days_till_order -= 1
        
        
        # ============================================================
        # 2. Initialize event flags for this timestep
        # ============================================================
        
        delivery_arrived = False
        order_placed = False
        day_rest_check = False
        
        
        # ============================================================
        # 3. Check whether an already placed order arrives today
        # ============================================================
        
        order_is_pending = not new_order
        
        if order_is_pending:
            delivery_due_today = (Delivery_sceduled - days_after_order) == 0
        
            if delivery_due_today:
                delivered_stock = max_delivery * Colli
        
                next_stock += delivered_stock
                actual_next_stock += delivered_stock
        
                days_after_order = 0
                new_order = True
                delivery_arrived = True
                day_rest_check = True
        
                delivery_amount[t] = delivered_stock
        
                # ----------------------------------------------------
                # Simulate missing delivered collis
                # ----------------------------------------------------
                if Flag_col_del_mis:
                    values = np.arange(0, max_delivery + 1)
        
                    # Exponentially decreasing probabilities
                    probs = np.exp(-values)
                    probs = probs / probs.sum()
        
                    Vault_coll = np.random.choice(values, p=probs)
                else:
                    Vault_coll = 0
        
                coll_stat[t] = Vault_coll
        
                # ----------------------------------------------------
                # Simulate missing products within collis
                # ----------------------------------------------------
                p_Colli_vault_prod = fsg.Colli_vault_prod(Vault_coll, max_delivery)
        
                if Flag_col_del_prod_mis:
                    Vault_coll_prod = np.random.choice(
                        np.arange(0, max_delivery + 1),
                        p=p_Colli_vault_prod
                    )
                else:
                    Vault_coll_prod = 0
        
                stock_missing[t] = abs(coll_stat[t] * Colli - Vault_coll_prod)

                #Add experation dates for deliveries
                Added_actual_stock = delivered_stock - stock_missing[t]
                #print("Added_actual_stock =", Added_actual_stock)
                pools.append([Exp_date] * Added_actual_stock)
                
        # ============================================================
        # 4. Decide whether to place a new order
        #    Only possible if no delivery arrived this timestep
        # ============================================================
        
        can_place_order = new_order and not delivery_arrived
        
        normal_order_needed = stock[t - 1] <= prod_min
        predicted_order_needed = days_till_order == 0
        
        if can_place_order and (normal_order_needed or predicted_order_needed):
            Delivery_sceduled = Delivery_time[t]
            days_after_order = 1
            new_order = False
            order_placed = True
        
            # Reset prediction countdown if the order was prediction-triggered
            if predicted_order_needed:
                days_till_order = None
                first_order_day = False
        
        elif order_is_pending and not delivery_arrived:
            # Order is still on its way
            days_after_order += 1
        
        
        # ============================================================
        # 5. Delivery_time is only non-zero when an order is placed
        # ============================================================
        
        if not order_placed:
            Delivery_time[t] = 0

            
################################################################################################################################

        
        # =======================================================================                               
        #Add randomized double_scanned in cost if scanned products (sold or rest)
        # =======================================================================
        
        if Flag_double_scanned == True:
            if sold[t] > 0 or rest[t] > 0:
                double_scanned[t] = np.random.choice(
                [0, 1, 2],
                p=fsg.double_scan_prob(sold[t], rest[t])
                )
                next_stock = next_stock - double_scanned[t] #Register extra scanned product from stock
                
            else:
                double_scanned[t] = 0 #Double but a well :)
                
        else:
            double_scanned[t] = 0

################################################################################################################################
        
        # =======================================================================                               
        #Add randomized Extra delivery
        # =======================================================================
        
        if Flag_col_del_extra == True:
            #There is a small chance of 1 or 2 extra collies
            extra_probs = np.array([0.995, 0.004, 0.001])  # corresponds to 0,1,2 extra collies
            extra_colli[t] = np.random.choice([0, 1, 2], p=extra_probs)
            max_extra = extra_colli[t]
            
            # Calculate missing products per colli if there are extra collies comming
            if max_extra > 0 and Flag_col_extra_del_prod_mis == True:
                volume = max_extra  # treat max_extra as "delivery size"
                scale = volume / max(volume, 1)  # avoid divide by zero
                missing_values = np.arange(0, 3)  # 0,1,2 missing products per extra colli
                missing_probs = np.exp(-scale * missing_values)
                missing_probs /= missing_probs.sum()  # normalize
            
                Vault_extra_coll_prod = np.random.choice(missing_values, p=missing_probs)
            else:
                Vault_extra_coll_prod = 0
        else:
            extra_colli[t] = 0
            Vault_extra_coll_prod = 0
        
            
        Extra_colli_stock[t] = int(extra_colli[t]) * Colli - int(Vault_extra_coll_prod)

        #Add experation dates for deliveries
        Added_actual_stock_EC = Extra_colli_stock[t]
        if Added_actual_stock_EC > 0:
            Added_actual_stock_EC = Extra_colli_stock[t]
            #print("Added_actual_stock_EC = ", Added_actual_stock_EC)
            pools.append([Exp_date] * Added_actual_stock_EC)
        

################################################################################################################################

        # =======================================================================                                   
        #Daily overload products from deliveries
        # =======================================================================                               
        if Flagdayremnants == True and day_rest_check == True: # Only perform a check in case you get a delivery!
            if (actual_next_stock + Extra_colli_stock[t] - stock_missing[t]) > prod_max:
                next_stock = actual_next_stock + Extra_colli_stock[t] - stock_missing[t]
                day_rest_cor[t] = 1

################################################################################################################################

        # ====================================================================================================================                               
        # stock corrections
        ###Manual stock correctie In case stock becomes negative you correct this product ones every 7 days in the evening preferably :)
        # Check every 7 days on wensdays, (two days apart from zerotelling), 
        #done in the evening after delivery is processed on wensdays (thus missing collies and products from deliveries from that day are also taken into account!)
        # ====================================================================================================================                               
        if Flagstockcorrecties == True:
            #print("prod = ",product_id)
            if stock[t-1] < 0:
                #print("First time neg stock t =", t)
                #print("t =", t)
                stock_correctie = True
                
            if (t+2) % 7 == 0 and t <= T and stock_correctie == True: 
                #print("stock correctie at t =",t)
                next_stock = actual_next_stock + Extra_colli_stock[t] - stock_missing[t]
                stock_cor[t] = 1 
                stock_correctie = False
                             

################################################################################################################################        
        #Update stock shown in system 
        stock.append(next_stock)
        #Update the actual stock 
        actual_stock[t] = actual_next_stock + Extra_colli_stock[t] - stock_missing[t]


################################################################################################################################        

        # =======================================================================                                   
        #Keep track of experation dates of each item
        # =======================================================================                               
        pools = fsg.update_pools(
            t=t,
            pools=pools,
            Initial_pool=Initial_pool,
            sold=sold[t],
            theft=theft[t],
            rest=rest[t]
        )

        #Update experation date of each individual items 
        pools = fsg.subtract_one(pools)

                    

        #print("t =", t)
        #print("pools = ", pools)
        #print("actual_stock[t] = ", actual_stock[t])
        #print("")
        
        #Keep track of expired items 
        expired_list, counted_negative_pools = fsg.check_negative_pools(pools, counted_negative_pools)
        expired_products[t] = expired_list 
        #print("t =", t)
        #print("expired_list =", expired_list)

        #print("pools =", pools)
        #print("expired_list =", expired_items)
        #print("")
        #print("")
             
        

################################################################################################################################                
        # ====================================================================================================================                               
        #Stock prediction pmf    
        # ====================================================================================================================                               
        if Flagorderprediction_pmf == True:

            manual_check_t = (
                zero_t[t] == 1
                or stock_cor[t] == 1
                or day_rest_cor[t] == 1
            )
            
            (base_prior_pmf, states, history,
             draws, multi_draws, base_history,
             pred_q10, pred_q50, pred_q90, time_to_order
            ) = pmf_up_script.stock_prediction(
                t,
                Idx_pred,
                stock,
                sold,
                delivery_amount,
                rng,
                pred_t,
                avg_delivery_time,
                window,
                lag,
                decay,
                rho,
                sigma,
                base_prior_pmf,
                states,
                history,
                draws,
                multi_draws,
                base_history,
                pred_q10,
                pred_q50,
                pred_q90,
                manual_check=manual_check_t,
                reset_on_manual_check=True,
                n_sims=1000, #number of simulations to run
                horizon=500, #Set amount of draws
                order_quantile="q50" #order based on which prediction
            )
        


        
        # ====================================================================================================================                               
        #Stock prediction Dirichlet  
        # ====================================================================================================================                                
        if Flagorderprediction_dir == True: 
            manual_check_t = (
                zero_t[t] == 1
                or stock_cor[t] == 1
                or day_rest_cor[t] == 1)

            # manual_check_del = (
            #     stock_cor[t] == 1
            #     or day_rest_cor[t] == 1)
    
    
            
            (alpha, states, alpha_base, 
            history, draws, multi_draws, 
            alpha_history, pred_q10, pred_q50, 
            pred_q90, time_to_order
            ) = Dir_script.stock_prediction_dirichlet(
                t=t,
                Idx_pred=Idx_pred,
                stock=stock,
                sold=sold,
                delivery_amount=delivery_amount,
                rng=rng,
                pred_t=pred_t,
                avg_delivery_time=avg_delivery_time,
                window=window,
                lag=lag,
                decay=decay,
                alpha_decay=0.995,
                obs_strength=1.0,
                alpha=alpha,
                states=states,
                alpha_base=alpha_base,
                history=history,
                draws=draws,
                multi_draws=multi_draws,
                alpha_history=alpha_history,
                pred_q10=pred_q10,
                pred_q50=pred_q50,
                pred_q90=pred_q90,
                manual_check=manual_check_t,
                reset_on_manual_check=True,
                horizon=400, #set amount of draws
                n_sims=1000, #number of simulations to run
                order_quantile="q50" #order based on which prediction
            )
        

        # ====================================================================================================================                               
        #Compute uncertaintly for Stock prediction (pmf or Dirichlet)  
        # ====================================================================================================================                                
        if Flagorderprediction_pmf == True or Flagorderprediction_dir == True:
            #Determine the predictions
            #print(manual_check_t)
            if t > Idx_pred:
                if manual_check_t:
                    stock_q10[t] = actual_stock[t]
                    stock_q50[t] = actual_stock[t]
                    stock_q90[t] = actual_stock[t]
                    delivered_stock_sum = 0
                    #print(stock_q10[t])
                    #print(actual_stock[t])
                    #print("check")
                    #print(t)
                    #print(pred_q10[t])
                
                elif manual_check_t == False:
                    for k in range(1, t + 1):
                        if len(pred_q10[t - k]) > 0:
                            stock_q10[t] = int(pred_q10[t - k][0][k - 1])
                            stock_q50[t] = int(pred_q50[t - k][0][k - 1])
                            stock_q90[t] = int(pred_q90[t - k][0][k - 1])
                            
                            #print(t)
                            #print(stock_q10[t])
                            break
                    if delivery_amount[t] > 0:
                        delivered_stock_sum += delivered_stock
                        # stock_q10[t] += delivered_stock_sum
                        # stock_q50[t] += delivered_stock_sum
                        # stock_q90[t] += delivered_stock_sum
                    
                    stock_q10[t] += delivered_stock_sum
                    stock_q50[t] += delivered_stock_sum
                    stock_q90[t] += delivered_stock_sum

                
                                        
                
            
            


    
    # ====================================================================================================================                                  
    #Add all variables to a dataframe
    # ====================================================================================================================  
    
    df_product = pd.DataFrame({
        "Product": [product_id]*T,
        "time": list(range(1, T+1)),
        "theft": theft,
        "Price": [price]*T,
        "day_of_week": day_of_week,
        "season": season,
        "stock": stock,
        "sold": sold,
        "Rest": rest,
        "prod_max": prod_max,
        "prod_min":prod_min,
        "max_delivery": max_delivery,
        "stock missing (C_M+P_M_C)": stock_missing,
        "Double scanned": double_scanned,
        "Colli extra (EX_C+P_M_C)": Extra_colli_stock,
        "actual_stock": actual_stock,
        "Colli": [Colli]*T,
        "day rest correctie": day_rest_cor, 
        "stock correctie": stock_cor,
        "zerotelling": zero_t,
        "Delivery time": Delivery_time,
        "delivery_amount": delivery_amount,
        "stock_q10": stock_q10,
        "stock_q50": stock_q50,
        "stock_q90": stock_q90, 
        "expired_products":expired_products,
    })

    all_products.append(df_product)
    
    empty_row = pd.DataFrame({
    "Product": [None],
    "time": [None],
    "theft": [None],
    "Price": [None],
    "day_of_week": [None],
    "season": [None],
    "stock": [None],
    "sold": [None],
    "Rest": [None],
    "prod_max": [None],
    "prod_min":[None],
    "max_delivery": [None],
    "stock missing (C_M+P_M_C)": [None],
    "Double scanned": [None],
    "Colli extra (EX_C+P_M_C)": [None],
    "actual_stock": [None],
    "Colli": [None],
    "day rest correctie": [None],
    "stock correctie": [None],
    "zerotelling": [None],
    "Delivery time": [None],
    "delivery_amount": [None],
    "stock_q10": [None],
    "stock_q50": [None],
    "stock_q90": [None],
    "expired_products": [None],
    })
    all_products.append(empty_row)

# Combine dataset
df_no_pred = pd.concat(all_products, ignore_index=True)

#Save data from first product 
#one_product = df_dir[df_dir["Product"] == 0]


product_id =  0
product_id =  1
product_id =  2
product_id =  3
product_id =  4
product_id =  5
product_id =  6
product_id =  7
product_id =  8
product_id =  9


In [8]:
df_no_pred[150:200]
# one_product = df_dir[df_dir["Product"] == 1]
# #one_product[100:150]

# one_product[one_product["expired_products"] != 0]

,Product,time,theft,Price,day_of_week,season,stock,sold,Rest,prod_max,...,Colli,day rest correctie,stock correctie,zerotelling,Delivery time,delivery_amount,stock_q10,stock_q50,stock_q90,expired_products
150,0,151,0,18,Sunday,spring,10,1,0,18,...,8,0,0,0,0,0,0,0,0,0
151,0,152,0,18,Monday,summer,10,0,0,18,...,8,0,0,0,0,0,0,0,0,0
152,0,153,0,18,Tuesday,summer,8,2,0,18,...,8,0,0,0,0,0,0,0,0,0
153,0,154,0,18,Wednesday,summer,7,1,0,18,...,8,0,0,0,0,0,0,0,0,0
154,0,155,0,18,Thursday,summer,7,0,0,18,...,8,0,0,0,0,0,0,0,0,0
155,0,156,0,18,Friday,summer,5,1,1,18,...,8,0,0,0,0,0,0,0,0,0
156,0,157,2,18,Saturday,summer,3,0,0,18,...,8,0,0,0,0,0,0,0,0,3
157,0,158,0,18,Sunday,summer,1,2,0,18,...,8,0,0,0,0,0,0,0,0,0
158,0,159,0,18,Monday,summer,0,1,0,18,...,8,0,0,0,3,0,0,0,0,0
159,0,160,0,18,Tuesday,summer,0,0,0,18,...,8,0,0,0,0,0,0,0,0,0


In [5]:
#Save to csv
#df_no_pred.to_csv("df_no_pred.csv", index=False)
#df_pmf.to_csv("df_pmf_q50.csv", index=False)
#df_dir.to_csv("df_dir_test.csv", index=False)
df_no_pred.to_csv("df_no_pred_test.csv", index=False)

# Plot 1 product

In [51]:
import pandas as pd
import matplotlib.pyplot as plt

# Clean separator rows
df_clean = df.replace("", pd.NA).dropna(subset=["Product"])

# Convert to numeric
cols = ["time", "Vooraard", "Verkocht", "theft", "Rest", "Product"]
df_clean[cols] = df_clean[cols].apply(pd.to_numeric)

# Select ONE product
product_id = 35

g = df_clean[df_clean["Product"] == product_id].sort_values("time")

plt.figure(figsize=(9, 5))

shift = 1

# --- Stacked bars (shifted left) ---
plt.bar(
    g["time"] - shift,
    g["Verkocht"],
    color="blue",
    alpha=0.3,
    label="Verkocht"
)

plt.bar(
    g["time"] - shift,
    g["theft"],
    bottom=g["Verkocht"],
    color="red",
    alpha=0.3,
    label="Diefstal"
)

plt.bar(
    g["time"] - shift,
    g["Rest"],
    bottom=g["Verkocht"] + g["theft"],
    color="green",
    alpha=0.3,
    label="Rest"
)

# --- Vooraard line (on top) ---
plt.plot(
    g["time"],
    g["Vooraard"],
    color="black",
    linewidth=2.5,
    marker="o",
    label="Vooraad"
)

plt.xlabel("Dag")
plt.ylabel("Vooraad")
plt.title(f"Voorraad systeem (product {product_id})")
plt.grid(axis="y")
plt.yticks(range(0, 21, 2))
#plt.xticks(g["time"])
plt.legend()
#plt.savefig("Voorraad product 9")

# --- ANNOTATIONS FOR CORRECTIONS ---
y_offset = 2  # vertical space above the line for text

# --- ANNOTATIONS FOR CORRECTIONS (STACKED) ---
base_offset = 2
step_offset = 1.2

for _, row in g.iterrows():
    t = row["time"]
    y = row["Vooraard"]

    level_above = 0
    level_below = 0

    if row["Dag rest correctie"] == 1:
        plt.annotate(
            "Dag restanten scan",
            xy=(t, y),
            xytext=(t, y + base_offset + level_above * step_offset),
            arrowprops=dict(arrowstyle="->", color="purple"),
            ha="center",
            fontsize=9,
            color="purple"
        )
        level_above += 1

    if row["Voorraad correctie"] == 1:
        plt.annotate(
            "Voorraad correctie scan",
            xy=(t, y),
            xytext=(t, y + base_offset + level_above * step_offset),
            arrowprops=dict(arrowstyle="->", color="orange"),
            ha="center",
            fontsize=9,
            color="orange"
        )
        level_above += 1

    if row["Nulltelling"] == 1:
        plt.annotate(
            "Nulltelling correctie",
            xy=(t, y),
            xytext=(t, y - base_offset - level_below * step_offset),
            arrowprops=dict(arrowstyle="->", color="red"),
            ha="center",
            fontsize=9,
            color="red"
        )
        level_below += 1




plt.show()

KeyError: "['Vooraard', 'Verkocht'] not in index"

# Backup

## Model

In [ ]:
# import Functions_stock_generation as fsg # Import stock generation functions
# import Initialize_product as ip # Call Variables from initialization
# from pmf_stock_pred import pmf_stock_predictions as pmf_script# Call functions from pmf prediction script



# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt


# #############################################Create data set#############################################

# #Set seed for random generations
# np.random.seed(20)
# #Set seed for pmf prediction
# rng = np.random.default_rng(42)
            
# #Initialize dataframe and time periods
# all_products = []
# columns = ["Product", "time", "theft", "Price", "Size",
#            "stock", "sold", "Rest", "Double scanned", "Extra colli",
#            "Colli", "zerotelling"]

# empty_row = pd.DataFrame([{col: "" for col in columns}]) # create empty row for readability
# T = 365*2  # total time periods 2 years!
# Shelf_length = 1 # 1 meters
# nr_products = 1000 #At 100 (but take 1000 to be sure!) you should be fine with your boxplots! no need to do more. 
# Idx_pred = int(np.round(T/2)) #Set Idx for stock prediction 


# #initialize variables for short and long timescales pmf updating 
# window = 28 #4 weeks, trend (note this can matter a lot)
# lag = 50 #time scale for rolling window base_pmf
# decay = 0.9 #decay used for updating the pmf
# rho = 0.03 #parameter for updating the base_pmf
# sigma = 0.8 #parameter for updating the base_pmf


# base_prior_pmf = 0 #for updating the base_pmf
# states = 0 #for updating the states in base_pmf
# history = [] #keep track of pmf history
# draws = [] #keep track of single draws
# multi_draws = [] #keep track of all draws
# base_history = [] #keep track of base_pmf history 


# #history = []
# #draws = []
# #multi_draws = []
# alpha_history = []
        


# #Calculations for size (needed for theft later on??)
# max_lenght, max_width, max_height = 0.25, 0.25, 0.25
# min_lenght, min_width, min_height = 0.1, 0.1, 0.1
# min_size = min_lenght * min_width * min_height 
# max_size = max_lenght * max_width * max_height 


# #Flags

# Flag_col_del_mis = True # missing collies
# Flag_col_del_prod_mis = True #Missing products within collies (normal delivery only)

# Flag_col_del_extra = True # Not registerd extra colli delivery 
# Flag_col_extra_del_prod_mis = True # Missing products within extra collies
# Flag_double_scanned = True # Products accidently scanned double or triple

# Flagzerotelling = True # Perform a zerotelling
# Flagstockcorrecties = True # Perform stock correctie
# Flagdayremnants = True # Perform checks in case the colli from the delivery does not fit in the shelf 
# Flagorderprediction = False # Predict when to order new products 

# #################################################Start generation of data###############################################


# for product_id in range(0, nr_products): #Loop over alll products
#     prod_init = ip.Initialize_prod(T, Shelf_length) 
#     price = prod_init["price"]
#     theft = prod_init["theft"]
#     sold = prod_init["sold"]
#     rest = prod_init["rest"]
#     max_prod_rows = prod_init["max_prod_rows"]
#     size = prod_init["size"]
#     prod_max = prod_init["prod_max"]
#     prod_min = prod_init["prod_min"]
#     Colli = prod_init["Colli"]
#     max_delivery = prod_init["max_delivery"]
#     Delivery_time = prod_init["Delivery_time"]
#     days_after_order = prod_init["days_after_order"]
#     Delivery_scheduled = prod_init["Delivery_scheduled"]
#     new_order = prod_init["new_order"]
#     delivery_amount = prod_init["delivery_amount"]
#     double_scanned = prod_init["double_scanned"]
#     extra_colli = prod_init["extra_colli"]
#     Extra_colli_stock = prod_init["Extra_colli_stock"]
#     coll_stat = prod_init["coll_stat"]
#     stock_missing = prod_init["stock_missing"]
#     zerotell = prod_init["zerotell"]
#     zero_t = prod_init["zero_t"]
#     stock = prod_init["stock"]
#     actual_stock = prod_init["actual_stock"]
#     stock_correctie = prod_init["stock_correctie"]
#     stock_cor = prod_init["stock_cor"]
#     day_rest_cor = prod_init["day_rest_cor"]
#     pred_t = prod_init["pred_t"]
#     avg_delivery_time = prod_init["avg_delivery_time"] 
#     days_till_order = prod_init["days_till_order"]
#     Frist_order_day = prod_init["Frist_order_day"]
#     time_to_order = prod_init["time_to_order"]
    

#     # ====================================================================================================================                               
#     #Loop over every timestep 
#     # ====================================================================================================================                               
#     for t in range(1, T):

#         #Precompute actual stock
#         actual_next_stock = (
#             actual_stock[t-1]
#             - sold[t]
#             - theft[t]
#             - rest[t]
#             #+ Vault_coll_prod # In case colli was missing 1 product!
#             )

#         if actual_next_stock < 0:
#             sold[t], theft[t], rest[t] = fsg.allocate_stock(
#                 actual_stock[t-1],
#                 sold[t],
#                 theft[t],
#                 rest[t]
#             )

#             #Correct actual stock
#             actual_next_stock = (
#                 actual_stock[t-1]
#                 - sold[t]
#                 - theft[t]
#                 - rest[t]
#                 #+ Vault_coll_prod # In case colli was missing 1 product!
#                 )
        
#         #Compute next stock in system with precomputed losses
#         next_stock = (
#             stock[t-1]
#             - sold[t]
#             - theft[t]
#             - rest[t]
#             #+ Vault_coll_prod # In case colli was missing 1 product!
#             )
        
#         # ====================================================================================================================                               
#         #Manual zero-telling (System says stock the size of a Colli or more is present but reality says we have no products)####
#         # Check every 7 days, 
#         #done in the morning!
#         # ====================================================================================================================                               
#         if t % 7 == 0 and t <= T and Flagzerotelling == True: 
#             #print("zero telling mogelijkheid t =" , t)
            
#             if actual_stock[t-1] == 0:
#                 #print("product_id =", product_id)
#                 #print("Correction t =",t)
#                 next_stock = 0
#                 zero_t[t] = 1
#                 sold[t] =0 
#                 theft[t] = 0
#                 rest[t] = 0
#                 actual_next_stock = 0
#                 #print("zero telling t =", t)
#         else:
#             zero_t[t] = 0
        
# ################################################################################################################
#         # ============================================================
#         # 1. Update prediction-based order countdown
#         # ============================================================
        
#         if not np.isnan(time_to_order):
#             # A new predicted order moment was found
#             days_till_order = int(time_to_order) - 1
#             first_order_day = True
        
#         elif days_till_order is not None and days_till_order > 0:
#             # Count down toward predicted order day
#             days_till_order -= 1
        
        
#         # ============================================================
#         # 2. Initialize event flags for this timestep
#         # ============================================================
        
#         delivery_arrived = False
#         order_placed = False
#         day_rest_check = False
        
        
#         # ============================================================
#         # 3. Check whether an already placed order arrives today
#         # ============================================================
        
#         order_is_pending = not new_order
        
#         if order_is_pending:
#             delivery_due_today = (Delivery_sceduled - days_after_order) == 0
        
#             if delivery_due_today:
#                 delivered_stock = max_delivery * Colli
        
#                 next_stock += delivered_stock
#                 actual_next_stock += delivered_stock
        
#                 days_after_order = 0
#                 new_order = True
#                 delivery_arrived = True
#                 day_rest_check = True
        
#                 delivery_amount[t] = delivered_stock
        
#                 # ----------------------------------------------------
#                 # Simulate missing delivered collis
#                 # ----------------------------------------------------
#                 if Flag_col_del_mis:
#                     values = np.arange(0, max_delivery + 1)
        
#                     # Exponentially decreasing probabilities
#                     probs = np.exp(-values)
#                     probs = probs / probs.sum()
        
#                     Vault_coll = np.random.choice(values, p=probs)
#                 else:
#                     Vault_coll = 0
        
#                 coll_stat[t] = Vault_coll
        
#                 # ----------------------------------------------------
#                 # Simulate missing products within collis
#                 # ----------------------------------------------------
#                 p_Colli_vault_prod = fsg.Colli_vault_prod(Vault_coll, max_delivery)
        
#                 if Flag_col_del_prod_mis:
#                     Vault_coll_prod = np.random.choice(
#                         np.arange(0, max_delivery + 1),
#                         p=p_Colli_vault_prod
#                     )
#                 else:
#                     Vault_coll_prod = 0
        
#                 stock_missing[t] = abs(coll_stat[t] * Colli - Vault_coll_prod)
        
        
#         # ============================================================
#         # 4. Decide whether to place a new order
#         #    Only possible if no delivery arrived this timestep
#         # ============================================================
        
#         can_place_order = new_order and not delivery_arrived
        
#         normal_order_needed = stock[t - 1] <= prod_min
#         predicted_order_needed = days_till_order == 0
        
#         if can_place_order and (normal_order_needed or predicted_order_needed):
#             Delivery_sceduled = Delivery_time[t]
#             days_after_order = 1
#             new_order = False
#             order_placed = True
        
#             # Reset prediction countdown if the order was prediction-triggered
#             if predicted_order_needed:
#                 days_till_order = None
#                 first_order_day = False
        
#         elif order_is_pending and not delivery_arrived:
#             # Order is still on its way
#             days_after_order += 1
        
        
#         # ============================================================
#         # 5. Delivery_time is only non-zero when an order is placed
#         # ============================================================
        
#         if not order_placed:
#             Delivery_time[t] = 0

            
# ################################################################################################################################

        
#         # =======================================================================                               
#         #Add randomized double_scanned in cost if scanned products (sold or rest)
#         # =======================================================================
        
#         if Flag_double_scanned == True:
#             if sold[t] > 0 or rest[t] > 0:
#                 double_scanned[t] = np.random.choice(
#                 [0, 1, 2],
#                 p=fsg.double_scan_prob(sold[t], rest[t])
#                 )
#                 next_stock = next_stock - double_scanned[t] #Register extra scanned product from stock
                
#             else:
#                 double_scanned[t] = 0 #Double but a well :)
                
#         else:
#             double_scanned[t] = 0
        

#         if Flag_col_del_extra == True:
#             #There is a small chance of 1 or 2 extra collies
#             extra_probs = np.array([0.995, 0.004, 0.001])  # corresponds to 0,1,2 extra collies
#             extra_colli[t] = np.random.choice([0, 1, 2], p=extra_probs)
#             max_extra = extra_colli[t]
            
#             # Calculate missing products per colli if there are extra collies comming
#             if max_extra > 0 and Flag_col_extra_del_prod_mis == True:
#                 volume = max_extra  # treat max_extra as "delivery size"
#                 scale = volume / max(volume, 1)  # avoid divide by zero
#                 missing_values = np.arange(0, 3)  # 0,1,2 missing products per extra colli
#                 missing_probs = np.exp(-scale * missing_values)
#                 missing_probs /= missing_probs.sum()  # normalize
            
#                 Vault_extra_coll_prod = np.random.choice(missing_values, p=missing_probs)
#             else:
#                 Vault_extra_coll_prod = 0
#         else:
#             extra_colli[t] = 0
#             Vault_extra_coll_prod = 0
        
            
#         Extra_colli_stock[t] = int(extra_colli[t]) * Colli - int(Vault_extra_coll_prod)
            
#         # =======================================================================                                   
#         #Daily overload products from deliveries
#         # =======================================================================                               
#         if Flagdayremnants == True and day_rest_check == True: # Only perform a check in case you get a delivery!
#             if (actual_next_stock + Extra_colli_stock[t] - stock_missing[t]) > prod_max:
#                 next_stock = actual_next_stock + Extra_colli_stock[t] - stock_missing[t]
#                 day_rest_cor[t] = 1

#         # ====================================================================================================================                               
#         # stock corrections
#         ###Manual stock correctie In case stock becomes negative you correct this product ones every 7 days in the evening preferably :)
#         # Check every 7 days on wensdays, (two days apart from zerotelling), 
#         #done in the evening after delivery is processed on wensdays (thus missing collies and products from deliveries from that day are also taken into account!)
#         # ====================================================================================================================                               
#         if Flagstockcorrecties == True:
#             #print("prod = ",product_id)
#             if stock[t-1] < 0:
#                 #print("First time neg stock t =", t)
#                 #print("t =", t)
#                 stock_correctie = True
                
#             if (t+2) % 7 == 0 and t <= T and stock_correctie == True: 
#                 #print("stock correctie at t =",t)
#                 next_stock = actual_next_stock + Extra_colli_stock[t] - stock_missing[t]
#                 stock_cor[t] = 1 
#                 stock_correctie = False
                             

#         #Update stock shown in system 
#         stock.append(next_stock)
#         #Update the actual stock 
#         actual_stock[t] = actual_next_stock + Extra_colli_stock[t] - stock_missing[t]

#         # ====================================================================================================================                               
#         #Stock prediction algo    
#         # ====================================================================================================================                               
#         if Flagorderprediction == True:
#             base_prior_pmf, states, history, draws, multi_draws, base_history, time_to_order = pmf_script.stock_prediction(
#                 t, 
#                 Idx_pred,
#                 stock,
#                 sold,
#                 delivery_amount,
#                 rng,
#                 pred_t,
#                 avg_delivery_time,
#                 window,
#                 lag, 
#                 decay,
#                 rho,
#                 sigma,
#                 base_prior_pmf,
#                 states,
#                 history,
#                 draws,
#                 multi_draws,
#                 base_history,)

#             # if time_to_order is not np.nan:
#             #     print(t)
#             #     print(time_to_order)
#             #print(multi_draws)
#             #print(t)
#             #print(time_to_order)
#             #print(base_prior_pmf)
#             #print(time_to_order)

            
#     # ====================================================================================================================                                  
#     #Add all variables to a dataframe
#     # ====================================================================================================================  
    
#     df_product = pd.DataFrame({
#         "Product": [product_id]*T,
#         "time": list(range(1, T+1)),
#         "theft": theft,
#         "Price": [price]*T,
#         "stock": stock,
#         "sold": sold,
#         "Rest": rest,
#         "prod_max": prod_max,
#         "prod_min":prod_min,
#         "max_delivery": max_delivery,
#         "stock missing (C_M+P_M_C)": stock_missing,
#         "Double scanned": double_scanned,
#         "Colli extra (EX_C+P_M_C)": Extra_colli_stock,
#         "actual_stock": actual_stock,
#         "Colli": [Colli]*T,
#         "day rest correctie": day_rest_cor, 
#         "stock correctie": stock_cor,
#         "zerotelling": zero_t,
#         "Delivery time": Delivery_time,
#         "delivery_amount": delivery_amount,
#     })

#     all_products.append(df_product)
    
#     empty_row = pd.DataFrame({
#     "Product": [None],
#     "time": [None],
#     "theft": [None],
#     "Price": [None],
#     "stock": [None],
#     "sold": [None],
#     "Rest": [None],
#     "prod_max": [None],
#     "prod_min":[None],
#     "max_delivery": [None],
#     "stock missing (C_M+P_M_C)": [None],
#     "Double scanned": [None],
#     "Colli extra (EX_C+P_M_C)": [None],
#     "actual_stock": [None],
#     "Colli": [None],
#     "day rest correctie": [None],
#     "stock correctie": [None],
#     "zerotelling": [None],
#     "Delivery time": [None],
#     "delivery_amount": [None],
#     })
#     all_products.append(empty_row)

# # Combine dataset
# df_no_pred = pd.concat(all_products, ignore_index=True)

# #Save data from first product 
# one_product = df_no_pred[df_no_pred["Product"] == 0]

In [ ]:
# import Functions_stock_generation as fsg # Import stock generation functions
# import Initialize_product as ip # Call Variables from initialization
# from pmf_stock_pred import pmf_stock_predictions as pmf_script# Call functions from pmf prediction script



# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt


# #############################################Create data set#############################################

# #Set seed for random generations
# np.random.seed(25)
# #Set seed for pmf prediction
# rng = np.random.default_rng(42)
            
# #Initialize dataframe and time periods
# all_products = []
# columns = ["Product", "time", "theft", "Price", "Size",
#            "stock", "sold", "Rest", "Double scanned", "Extra colli",
#            "Colli", "zerotelling"]

# empty_row = pd.DataFrame([{col: "" for col in columns}]) # create empty row for readability
# T = 365*2  # total time periods 2 years!
# Shelf_length = 1 # 1 meters
# nr_products = 1000 #At 100 (but take 1000 to be sure!) you should be fine with your boxplots! no need to do more. 
# Idx_pred = int(np.round(T/2)) #Set Idx for stock prediction 


# #initialize variables for short and long timescales pmf updating 
# window = 28 #4 weeks, trend (note this can matter a lot)
# lag = 50 #time scale for rolling window base_pmf
# decay = 0.9 #decay used for updating the pmf
# rho = 0.03 #parameter for updating the base_pmf
# sigma = 0.8 #parameter for updating the base_pmf
# base_prior_pmf = 0 #for updating the base_pmf
# states = 0 #for updating the states in base_pmf
# history = [] #keep track of pmf history
# draws = [] #keep track of single draws
# multi_draws = [] #keep track of all draws
# base_history = [] #keep track of base_pmf history 

        


# #Calculations for size (needed for theft later on??)
# max_lenght, max_width, max_height = 0.25, 0.25, 0.25
# min_lenght, min_width, min_height = 0.1, 0.1, 0.1
# min_size = min_lenght * min_width * min_height 
# max_size = max_lenght * max_width * max_height 


# #Flags

# Flag_col_del_mis = True # missing collies
# Flag_col_del_prod_mis = True #Missing products within collies (normal delivery only)

# Flag_col_del_extra = True # Not registerd extra colli delivery 
# Flag_col_extra_del_prod_mis = True # Missing products within extra collies
# Flag_double_scanned = True # Products accidently scanned double or triple

# Flagzerotelling = True # Perform a zerotelling
# Flagstockcorrecties = True # Perform stock correctie
# Flagdayremnants = True # Perform checks in case the colli from the delivery does not fit in the shelf 
# Flagorderprediction = True # Predict when to order new products 

# #################################################Start generation of data###############################################


# for product_id in range(0, nr_products): #Loop over alll products
#     prod_init = ip.Initialize_prod(T, Shelf_length) 
#     price = prod_init["price"]
#     theft = prod_init["theft"]
#     sold = prod_init["sold"]
#     rest = prod_init["rest"]
#     max_prod_rows = prod_init["max_prod_rows"]
#     size = prod_init["size"]
#     prod_max = prod_init["prod_max"]
#     prod_min = prod_init["prod_min"]
#     Colli = prod_init["Colli"]
#     max_delivery = prod_init["max_delivery"]
#     Delivery_time = prod_init["Delivery_time"]
#     days_after_order = prod_init["days_after_order"]
#     Delivery_scheduled = prod_init["Delivery_scheduled"]
#     new_order = prod_init["new_order"]
#     delivery_amount = prod_init["delivery_amount"]
#     double_scanned = prod_init["double_scanned"]
#     extra_colli = prod_init["extra_colli"]
#     Extra_colli_stock = prod_init["Extra_colli_stock"]
#     coll_stat = prod_init["coll_stat"]
#     stock_missing = prod_init["stock_missing"]
#     zerotell = prod_init["zerotell"]
#     zero_t = prod_init["zero_t"]
#     stock = prod_init["stock"]
#     actual_stock = prod_init["actual_stock"]
#     stock_correctie = prod_init["stock_correctie"]
#     stock_cor = prod_init["stock_cor"]
#     day_rest_cor = prod_init["day_rest_cor"]
#     pred_t = prod_init["pred_t"]
#     avg_delivery_time = prod_init["avg_delivery_time"] 
#     #Still need to add these in Initialize!
#     days_till_order = None
#     Frist_order_day = True
#     time_to_order = np.nan
    
#     #Loop over every timestep 
#     for t in range(1, T):

#         #Precompute actual stock
#         actual_next_stock = (
#             actual_stock[t-1]
#             - sold[t]
#             - theft[t]
#             - rest[t]
#             #+ Vault_coll_prod # In case colli was missing 1 product!
#             )

#         if actual_next_stock < 0:
#             sold[t], theft[t], rest[t] = fsg.allocate_stock(
#                 actual_stock[t-1],
#                 sold[t],
#                 theft[t],
#                 rest[t]
#             )

#             #Correct actual stock
#             actual_next_stock = (
#                 actual_stock[t-1]
#                 - sold[t]
#                 - theft[t]
#                 - rest[t]
#                 #+ Vault_coll_prod # In case colli was missing 1 product!
#                 )
        
#         #Compute next stock in system with precomputed losses
#         next_stock = (
#             stock[t-1]
#             - sold[t]
#             - theft[t]
#             - rest[t]
#             #+ Vault_coll_prod # In case colli was missing 1 product!
#             )

#         ####Manual zero-telling (System says stock the size of a Colli or more is present but reality says we have no products)####
#         # Check every 7 days, 
#         #done in the morning!
#         if t % 7 == 0 and t <= T and Flagzerotelling == True: 
#             #print("zero telling mogelijkheid t =" , t)
            
#             if actual_stock[t-1] == 0:
#                 #print("product_id =", product_id)
#                 #print("Correction t =",t)
#                 next_stock = 0
#                 zero_t[t] = 1
#                 sold[t] =0 
#                 theft[t] = 0
#                 rest[t] = 0
#                 actual_next_stock = 0
#                 #print("zero telling t =", t)
#         else:
#             zero_t[t] = 0
        
# ################################################################################################################
#         if not np.isnan(time_to_order):
#             days_till_order = int(time_to_order) - 1
#             first_order_day = True
        
#             # print("first flagged")
#             # print(t)
#             # #print("time_to_order =", time_to_order)
#             # print("days_till_order = ",days_till_order)
        
#         elif days_till_order is not None and days_till_order > 0:
#             days_till_order -= 1
        
#             # print(t)
#             # print("days_till_order = ",days_till_order)
        
#         elif days_till_order == 0 and new_order == True:
#             #Here can order!
#             # print(t)
#             # print("order")
            
#             days_after_order = 1
#             new_order == True
        
#             # reset after ordering
#             days_till_order = None
#             first_order_day = False

            
# #############################################################################################################

        
#         delivery_arrived = False
#         order_placed = False 
#         day_rest_check = False
#         # Add extra stock delivery is bound to arrive
#         if new_order == False:
#             if (Delivery_sceduled - days_after_order) == 0:
#                 next_stock += max_delivery * Colli
#                 actual_next_stock += max_delivery * Colli
#                 days_after_order = 0
#                 new_order = True
#                 delivery_arrived = True
#                 day_rest_check = True
#                 delivery_amount[t] = max_delivery * Colli
#                 # print(t)
#                 # print("Order arrived!")


#                 if Flag_col_del_mis == True:
#                     values = np.arange(0, max_delivery + 1)
                
#                     # exponentially decreasing probabilities
#                     probs = np.exp(-values)
#                     probs = probs / probs.sum()
                
#                     Vault_coll = np.random.choice(values, p=probs)
#                 else:
#                     Vault_coll = 0
                
#                 coll_stat[t] = Vault_coll
    

#                 p_Colli_vault_prod = fsg.Colli_vault_prod(Vault_coll, max_delivery)

#                 if Flag_col_del_prod_mis:
#                     Vault_coll_prod = np.random.choice(
#                         np.arange(0, max_delivery + 1),
#                         p=p_Colli_vault_prod
#                     )
#                 else:
#                     Vault_coll_prod = 0
                    
    
#                 stock_missing[t] = np.abs(coll_stat[t]*Colli - Vault_coll_prod)
#                 # print("t =",t)
#                 # print("stock_missing[t] =", stock_missing[t])


        
#         #Make a order decision (which keeps track of the count when an order is made)
#         if delivery_arrived == False:
#             if new_order == True and stock[t-1] <= prod_min:
#                 Delivery_sceduled = Delivery_time[t]   
#                 days_after_order = 1
#                 new_order = False
#                 order_placed = True
#                 # print(t)
#                 # print("normal order")


                
# ######################################################################################################               
#             elif new_order == True and days_till_order == 0:
#                 Delivery_sceduled = Delivery_time[t]   
#                 days_after_order = 1
#                 new_order = False
#                 order_placed = True
#                 # print(t)
#                 # print("Delivery_sceduled = ", Delivery_sceduled)


# ######################################################################################################
            
#             elif new_order == False:
#                 days_after_order += 1
#                 # print(t)
#                 # print("days_after_order = ", days_after_order)
        
#         # Ensure delivery_time is set 0 zero, (only non-zero when an order is placed!)
#         if not order_placed:
#             Delivery_time[t] = 0

                       
                                        
#         #Add randomized double_scanned in cost if scanned products (sold or rest)
#         if Flag_double_scanned == True:
#             if sold[t] > 0 or rest[t] > 0:
#                 double_scanned[t] = np.random.choice(
#                 [0, 1, 2],
#                 p=fsg.double_scan_prob(sold[t], rest[t])
#                 )
#                 next_stock = next_stock - double_scanned[t] #Register extra scanned product from stock
                
#             else:
#                 double_scanned[t] = 0 #Double but a well :)
                
#         else:
#             double_scanned[t] = 0
        

#         if Flag_col_del_extra == True:
#             #There is a small chance of 1 or 2 extra collies
#             extra_probs = np.array([0.995, 0.004, 0.001])  # corresponds to 0,1,2 extra collies
#             extra_colli[t] = np.random.choice([0, 1, 2], p=extra_probs)
#             max_extra = extra_colli[t]
            
#             # Calculate missing products per colli if there are extra collies comming
#             if max_extra > 0 and Flag_col_extra_del_prod_mis == True:
#                 volume = max_extra  # treat max_extra as "delivery size"
#                 scale = volume / max(volume, 1)  # avoid divide by zero
#                 missing_values = np.arange(0, 3)  # 0,1,2 missing products per extra colli
#                 missing_probs = np.exp(-scale * missing_values)
#                 missing_probs /= missing_probs.sum()  # normalize
            
#                 Vault_extra_coll_prod = np.random.choice(missing_values, p=missing_probs)
#             else:
#                 Vault_extra_coll_prod = 0
#         else:
#             extra_colli[t] = 0
#             Vault_extra_coll_prod = 0
        
            
#         Extra_colli_stock[t] = int(extra_colli[t]) * Colli - int(Vault_extra_coll_prod)
            
            
#         #Daily overload products from deliveries
#         if Flagdayremnants == True and day_rest_check == True: # Only perform a check in case you get a delivery!
#             if (actual_next_stock + Extra_colli_stock[t] - stock_missing[t]) > prod_max:
#                 next_stock = actual_next_stock + Extra_colli_stock[t] - stock_missing[t]
#                 day_rest_cor[t] = 1

        
#         # stock corrections
        

#         ###Manual stock correctie In case stock becomes negative you correct this product ones every 7 days in the evening preferably :)
#         # Check every 7 days on wensdays, (two days apart from zerotelling), 
#         #done in the evening after delivery is processed on wensdays (thus missing collies and products from deliveries from that day are also taken into account!)

#         if Flagstockcorrecties == True:
#             #print("prod = ",product_id)
#             if stock[t-1] < 0:
#                 #print("First time neg stock t =", t)
#                 #print("t =", t)
#                 stock_correctie = True
                
#             if (t+2) % 7 == 0 and t <= T and stock_correctie == True: 
#                 #print("stock correctie at t =",t)
#                 next_stock = actual_next_stock + Extra_colli_stock[t] - stock_missing[t]
#                 stock_cor[t] = 1 
#                 stock_correctie = False
                             

#         #Update stock shown in system 
#         stock.append(next_stock)
#         #Update the actual stock 
#         actual_stock[t] = actual_next_stock + Extra_colli_stock[t] - stock_missing[t]

# ########################################Stock prediction algo##############################################################    
#         if Flagorderprediction == True:
#             #print(t)
#             base_prior_pmf, states, history, draws, multi_draws, base_history, time_to_order = pmf_script.stock_prediction(
#                 t, 
#                 Idx_pred,
#                 stock,
#                 sold,
#                 delivery_amount,
#                 rng,
#                 pred_t,
#                 avg_delivery_time,
#                 window,
#                 lag, 
#                 decay,
#                 rho,
#                 sigma,
#                 base_prior_pmf,
#                 states,
#                 history,
#                 draws,
#                 multi_draws,
#                 base_history,)

#             # if time_to_order is not np.nan:
#             #     print(t)
#             #     print(time_to_order)
#             #print(multi_draws)
#             #print(t)
#             #print(time_to_order)
#             #print(base_prior_pmf)
#             #print(time_to_order)

            
    
#     ########Add all variables to a dataframe################
#     df_product = pd.DataFrame({
#         "Product": [product_id]*T,
#         "time": list(range(1, T+1)),
#         "theft": theft,
#         "Price": [price]*T,
#         "stock": stock,
#         "sold": sold,
#         "Rest": rest,
#         "prod_max": prod_max,
#         "prod_min":prod_min,
#         "max_delivery": max_delivery,
#         "stock missing (C_M+P_M_C)": stock_missing,
#         "Double scanned": double_scanned,
#         "Colli extra (EX_C+P_M_C)": Extra_colli_stock,
#         "actual_stock": actual_stock,
#         "Colli": [Colli]*T,
#         "day rest correctie": day_rest_cor, 
#         "stock correctie": stock_cor,
#         "zerotelling": zero_t,
#         "Delivery time": Delivery_time,
#         "delivery_amount": delivery_amount,
#     })

#     all_products.append(df_product)
    
#     empty_row = pd.DataFrame({
#     "Product": [None],
#     "time": [None],
#     "theft": [None],
#     "Price": [None],
#     "stock": [None],
#     "sold": [None],
#     "Rest": [None],
#     "prod_max": [None],
#     "prod_min":[None],
#     "max_delivery": [None],
#     "stock missing (C_M+P_M_C)": [None],
#     "Double scanned": [None],
#     "Colli extra (EX_C+P_M_C)": [None],
#     "actual_stock": [None],
#     "Colli": [None],
#     "day rest correctie": [None],
#     "stock correctie": [None],
#     "zerotelling": [None],
#     "Delivery time": [None],
#     "delivery_amount": [None],
#     })
#     all_products.append(empty_row)

# # Combine dataset
# df_pred = pd.concat(all_products, ignore_index=True)

# #Save data from first product 
# one_product = df[df["Product"] == 0]

In [ ]:
# import Functions_stock_generation as fsg # Import stock generation functions
# import Initialize_product as ip # Call Variables from initialization
# from pmf_stock_pred import pmf_stock_predictions as pmf_script# Call functions from pmf prediction script



# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt


# #############################################Create data set#############################################

# #Set seed for random generations
# np.random.seed(25)
# #Set seed for pmf prediction
# rng = np.random.default_rng(42)
            
# #Initialize dataframe and time periods
# all_products = []
# columns = ["Product", "time", "theft", "Price", "Size",
#            "stock", "sold", "Rest", "Double scanned", "Extra colli",
#            "Colli", "zerotelling"]

# empty_row = pd.DataFrame([{col: "" for col in columns}]) # create empty row for readability
# T = 365*2  # total time periods 2 years!
# Shelf_length = 1 # 1 meters
# nr_products = 1000 #At 100 (but take 1000 to be sure!) you should be fine with your boxplots! no need to do more. 
# Idx_pred = int(np.round(T/2)) #Set Idx for stock prediction 


# #initialize variables for short and long timescales pmf updating 
# window = 28 #4 weeks, trend (note this can matter a lot)
# lag = 50 #time scale for rolling window base_pmf
# decay = 0.9 #decay used for updating the pmf
# rho = 0.03 #parameter for updating the base_pmf
# sigma = 0.8 #parameter for updating the base_pmf
# base_prior_pmf = 0 #for updating the base_pmf
# states = 0 #for updating the states in base_pmf
# history = [] #keep track of pmf history
# draws = [] #keep track of single draws
# multi_draws = [] #keep track of all draws
# base_history = [] #keep track of base_pmf history 

        


# #Calculations for size (needed for theft later on??)
# max_lenght, max_width, max_height = 0.25, 0.25, 0.25
# min_lenght, min_width, min_height = 0.1, 0.1, 0.1
# min_size = min_lenght * min_width * min_height 
# max_size = max_lenght * max_width * max_height 


# #Flags

# Flag_col_del_mis = True # missing collies
# Flag_col_del_prod_mis = True #Missing products within collies (normal delivery only)

# Flag_col_del_extra = True # Not registerd extra colli delivery 
# Flag_col_extra_del_prod_mis = True # Missing products within extra collies
# Flag_double_scanned = True # Products accidently scanned double or triple

# Flagzerotelling = True # Perform a zerotelling
# Flagstockcorrecties = True # Perform stock correctie
# Flagdayremnants = True # Perform checks in case the colli from the delivery does not fit in the shelf 
# Flagorderprediction = True # Predict when to order new products 

# #################################################Start generation of data###############################################


# for product_id in range(0, nr_products): #Loop over alll products
#     prod_init = ip.Initialize_prod(T, Shelf_length) 
#     price = prod_init["price"]
#     theft = prod_init["theft"]
#     sold = prod_init["sold"]
#     rest = prod_init["rest"]
#     max_prod_rows = prod_init["max_prod_rows"]
#     size = prod_init["size"]
#     prod_max = prod_init["prod_max"]
#     prod_min = prod_init["prod_min"]
#     Colli = prod_init["Colli"]
#     max_delivery = prod_init["max_delivery"]
#     Delivery_time = prod_init["Delivery_time"]
#     days_after_order = prod_init["days_after_order"]
#     Delivery_scheduled = prod_init["Delivery_scheduled"]
#     new_order = prod_init["new_order"]
#     delivery_amount = prod_init["delivery_amount"]
#     double_scanned = prod_init["double_scanned"]
#     extra_colli = prod_init["extra_colli"]
#     Extra_colli_stock = prod_init["Extra_colli_stock"]
#     coll_stat = prod_init["coll_stat"]
#     stock_missing = prod_init["stock_missing"]
#     zerotell = prod_init["zerotell"]
#     zero_t = prod_init["zero_t"]
#     stock = prod_init["stock"]
#     actual_stock = prod_init["actual_stock"]
#     stock_correctie = prod_init["stock_correctie"]
#     stock_cor = prod_init["stock_cor"]
#     day_rest_cor = prod_init["day_rest_cor"]
#     pred_t = prod_init["pred_t"]
#     avg_delivery_time = prod_init["avg_delivery_time"] 
#     #Still need to add these in Initialize!
#     days_till_order = None
#     Frist_order_day = True
#     time_to_order = np.nan
    
#     #Loop over every timestep 
#     for t in range(1, T):

#         #Precompute actual stock
#         actual_next_stock = (
#             actual_stock[t-1]
#             - sold[t]
#             - theft[t]
#             - rest[t]
#             #+ Vault_coll_prod # In case colli was missing 1 product!
#             )

#         if actual_next_stock < 0:
#             sold[t], theft[t], rest[t] = fsg.allocate_stock(
#                 actual_stock[t-1],
#                 sold[t],
#                 theft[t],
#                 rest[t]
#             )

#             #Correct actual stock
#             actual_next_stock = (
#                 actual_stock[t-1]
#                 - sold[t]
#                 - theft[t]
#                 - rest[t]
#                 #+ Vault_coll_prod # In case colli was missing 1 product!
#                 )
        
#         #Compute next stock in system with precomputed losses
#         next_stock = (
#             stock[t-1]
#             - sold[t]
#             - theft[t]
#             - rest[t]
#             #+ Vault_coll_prod # In case colli was missing 1 product!
#             )

#         ####Manual zero-telling (System says stock the size of a Colli or more is present but reality says we have no products)####
#         # Check every 7 days, 
#         #done in the morning!
#         if t % 7 == 0 and t <= T and Flagzerotelling == True: 
#             #print("zero telling mogelijkheid t =" , t)
            
#             if actual_stock[t-1] == 0:
#                 #print("product_id =", product_id)
#                 #print("Correction t =",t)
#                 next_stock = 0
#                 zero_t[t] = 1
#                 sold[t] =0 
#                 theft[t] = 0
#                 rest[t] = 0
#                 actual_next_stock = 0
#                 #print("zero telling t =", t)
#         else:
#             zero_t[t] = 0
        
# ################################################################################################################
#         if not np.isnan(time_to_order):
#             days_till_order = int(time_to_order) - 1
#             first_order_day = True
        
#             # print("first flagged")
#             # print(t)
#             # #print("time_to_order =", time_to_order)
#             # print("days_till_order = ",days_till_order)
        
#         elif days_till_order is not None and days_till_order > 0:
#             days_till_order -= 1
        
#             # print(t)
#             # print("days_till_order = ",days_till_order)
        
#         elif days_till_order == 0 and new_order == True:
#             #Here can order!
#             # print(t)
#             # print("order")
            
#             days_after_order = 1
#             new_order == True
        
#             # reset after ordering
#             days_till_order = None
#             first_order_day = False

            
# #############################################################################################################

        
#         delivery_arrived = False
#         order_placed = False 
#         day_rest_check = False
#         # Add extra stock delivery is bound to arrive
#         if new_order == False:
#             if (Delivery_sceduled - days_after_order) == 0:
#                 next_stock += max_delivery * Colli
#                 actual_next_stock += max_delivery * Colli
#                 days_after_order = 0
#                 new_order = True
#                 delivery_arrived = True
#                 day_rest_check = True
#                 delivery_amount[t] = max_delivery * Colli
#                 # print(t)
#                 # print("Order arrived!")


#                 if Flag_col_del_mis == True:
#                     values = np.arange(0, max_delivery + 1)
                
#                     # exponentially decreasing probabilities
#                     probs = np.exp(-values)
#                     probs = probs / probs.sum()
                
#                     Vault_coll = np.random.choice(values, p=probs)
#                 else:
#                     Vault_coll = 0
                
#                 coll_stat[t] = Vault_coll
    

#                 p_Colli_vault_prod = fsg.Colli_vault_prod(Vault_coll, max_delivery)

#                 if Flag_col_del_prod_mis:
#                     Vault_coll_prod = np.random.choice(
#                         np.arange(0, max_delivery + 1),
#                         p=p_Colli_vault_prod
#                     )
#                 else:
#                     Vault_coll_prod = 0
                    
    
#                 stock_missing[t] = np.abs(coll_stat[t]*Colli - Vault_coll_prod)
#                 # print("t =",t)
#                 # print("stock_missing[t] =", stock_missing[t])


        
#         #Make a order decision (which keeps track of the count when an order is made)
#         if delivery_arrived == False:
#             if new_order == True and stock[t-1] <= prod_min:
#                 Delivery_sceduled = Delivery_time[t]   
#                 days_after_order = 1
#                 new_order = False
#                 order_placed = True
#                 # print(t)
#                 # print("normal order")


                
# ######################################################################################################               
#             elif new_order == True and days_till_order == 0:
#                 Delivery_sceduled = Delivery_time[t]   
#                 days_after_order = 1
#                 new_order = False
#                 order_placed = True
#                 # print(t)
#                 # print("Delivery_sceduled = ", Delivery_sceduled)


# ######################################################################################################
            
#             elif new_order == False:
#                 days_after_order += 1
#                 # print(t)
#                 # print("days_after_order = ", days_after_order)
        
#         # Ensure delivery_time is set 0 zero, (only non-zero when an order is placed!)
#         if not order_placed:
#             Delivery_time[t] = 0

                       
                                        
#         #Add randomized double_scanned in cost if scanned products (sold or rest)
#         if Flag_double_scanned == True:
#             if sold[t] > 0 or rest[t] > 0:
#                 double_scanned[t] = np.random.choice(
#                 [0, 1, 2],
#                 p=fsg.double_scan_prob(sold[t], rest[t])
#                 )
#                 next_stock = next_stock - double_scanned[t] #Register extra scanned product from stock
                
#             else:
#                 double_scanned[t] = 0 #Double but a well :)
                
#         else:
#             double_scanned[t] = 0
        

#         if Flag_col_del_extra == True:
#             #There is a small chance of 1 or 2 extra collies
#             extra_probs = np.array([0.995, 0.004, 0.001])  # corresponds to 0,1,2 extra collies
#             extra_colli[t] = np.random.choice([0, 1, 2], p=extra_probs)
#             max_extra = extra_colli[t]
            
#             # Calculate missing products per colli if there are extra collies comming
#             if max_extra > 0 and Flag_col_extra_del_prod_mis == True:
#                 volume = max_extra  # treat max_extra as "delivery size"
#                 scale = volume / max(volume, 1)  # avoid divide by zero
#                 missing_values = np.arange(0, 3)  # 0,1,2 missing products per extra colli
#                 missing_probs = np.exp(-scale * missing_values)
#                 missing_probs /= missing_probs.sum()  # normalize
            
#                 Vault_extra_coll_prod = np.random.choice(missing_values, p=missing_probs)
#             else:
#                 Vault_extra_coll_prod = 0
#         else:
#             extra_colli[t] = 0
#             Vault_extra_coll_prod = 0
        
            
#         Extra_colli_stock[t] = int(extra_colli[t]) * Colli - int(Vault_extra_coll_prod)
            
            
#         #Daily overload products from deliveries
#         if Flagdayremnants == True and day_rest_check == True: # Only perform a check in case you get a delivery!
#             if (actual_next_stock + Extra_colli_stock[t] - stock_missing[t]) > prod_max:
#                 next_stock = actual_next_stock + Extra_colli_stock[t] - stock_missing[t]
#                 day_rest_cor[t] = 1

        
#         # stock corrections
        

#         ###Manual stock correctie In case stock becomes negative you correct this product ones every 7 days in the evening preferably :)
#         # Check every 7 days on wensdays, (two days apart from zerotelling), 
#         #done in the evening after delivery is processed on wensdays (thus missing collies and products from deliveries from that day are also taken into account!)

#         if Flagstockcorrecties == True:
#             #print("prod = ",product_id)
#             if stock[t-1] < 0:
#                 #print("First time neg stock t =", t)
#                 #print("t =", t)
#                 stock_correctie = True
                
#             if (t+2) % 7 == 0 and t <= T and stock_correctie == True: 
#                 #print("stock correctie at t =",t)
#                 next_stock = actual_next_stock + Extra_colli_stock[t] - stock_missing[t]
#                 stock_cor[t] = 1 
#                 stock_correctie = False
                             

#         #Update stock shown in system 
#         stock.append(next_stock)
#         #Update the actual stock 
#         actual_stock[t] = actual_next_stock + Extra_colli_stock[t] - stock_missing[t]

# ########################################Stock prediction algo##############################################################    
#         if Flagorderprediction == True:
#             #print(t)
#             base_prior_pmf, states, history, draws, multi_draws, base_history, time_to_order = pmf_script.stock_prediction(
#                 t, 
#                 Idx_pred,
#                 stock,
#                 sold,
#                 delivery_amount,
#                 rng,
#                 pred_t,
#                 avg_delivery_time,
#                 window,
#                 lag, 
#                 decay,
#                 rho,
#                 sigma,
#                 base_prior_pmf,
#                 states,
#                 history,
#                 draws,
#                 multi_draws,
#                 base_history,)

#             # if time_to_order is not np.nan:
#             #     print(t)
#             #     print(time_to_order)
#             #print(multi_draws)
#             #print(t)
#             #print(time_to_order)
#             #print(base_prior_pmf)
#             #print(time_to_order)

            
    
#     ########Add all variables to a dataframe################
#     df_product = pd.DataFrame({
#         "Product": [product_id]*T,
#         "time": list(range(1, T+1)),
#         "theft": theft,
#         "Price": [price]*T,
#         "stock": stock,
#         "sold": sold,
#         "Rest": rest,
#         "prod_max": prod_max,
#         "prod_min":prod_min,
#         "max_delivery": max_delivery,
#         "stock missing (C_M+P_M_C)": stock_missing,
#         "Double scanned": double_scanned,
#         "Colli extra (EX_C+P_M_C)": Extra_colli_stock,
#         "actual_stock": actual_stock,
#         "Colli": [Colli]*T,
#         "day rest correctie": day_rest_cor, 
#         "stock correctie": stock_cor,
#         "zerotelling": zero_t,
#         "Delivery time": Delivery_time,
#         "delivery_amount": delivery_amount,
#     })

#     all_products.append(df_product)
    
#     empty_row = pd.DataFrame({
#     "Product": [None],
#     "time": [None],
#     "theft": [None],
#     "Price": [None],
#     "stock": [None],
#     "sold": [None],
#     "Rest": [None],
#     "prod_max": [None],
#     "prod_min":[None],
#     "max_delivery": [None],
#     "stock missing (C_M+P_M_C)": [None],
#     "Double scanned": [None],
#     "Colli extra (EX_C+P_M_C)": [None],
#     "actual_stock": [None],
#     "Colli": [None],
#     "day rest correctie": [None],
#     "stock correctie": [None],
#     "zerotelling": [None],
#     "Delivery time": [None],
#     "delivery_amount": [None],
#     })
#     all_products.append(empty_row)

# # Combine dataset
# df_pred = pd.concat(all_products, ignore_index=True)

# #Save data from first product 
# one_product = df[df["Product"] == 0]

In [ ]:
# import Functions_stock_generation as fsg # Import stock generation functions
# import Initialize_product as ip # Call Variables from initialization
# from pmf_stock_pred import pmf_stock_predictions as pmf_script# Call functions from pmf prediction script



# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt


# #############################################Create data set#############################################

# #Set seed for random generations
# np.random.seed(25)
# #Set seed for pmf prediction)
# rng = np.random.default_rng(42)
            
# #Initialize dataframe and time periods
# all_products = []
# columns = ["Product", "time", "theft", "Price", "Size",
#            "stock", "sold", "Rest", "Double scanned", "Extra colli",
#            "Colli", "zerotelling"]

# empty_row = pd.DataFrame([{col: "" for col in columns}]) # create empty row for readability
# T = 365*2  # total time periods 2 years!
# Shelf_length = 1 # 1 meters
# nr_products = 10 #At 100 you should be fine with your boxplots! no need to do more. 
# Idx_pred = int(np.round(T/2)) #Set Idx for stock prediction 


# #initialize variables for short and long timescales pmf updating 
# window = 28 #4 weeks, trend (note this can matter a lot)
# lag = 50 #time scale for rolling window base_pmf
# decay = 0.9 #decay used for updating the pmf
# rho = 0.03 #parameter for updating the base_pmf
# sigma = 0.8 #parameter for updating the base_pmf
# base_prior_pmf = 0 #for updating the base_pmf
# states = 0 #for updating the states in base_pmf
# history = []
# draws = []
# multi_draws = []
# base_history = []
# time_to_order = np.nan
        


# #Calculations for size (needed for theft later on??)
# max_lenght, max_width, max_height = 0.25, 0.25, 0.25
# min_lenght, min_width, min_height = 0.1, 0.1, 0.1
# min_size = min_lenght * min_width * min_height 
# max_size = max_lenght * max_width * max_height 


# #Flags

# Flag_col_del_mis = True # missing collies
# Flag_col_del_prod_mis = True #Missing products within collies (normal delivery only)

# Flag_col_del_extra = True # Not registerd extra colli delivery 
# Flag_col_extra_del_prod_mis = True # Missing products within extra collies
# Flag_double_scanned = True # Products accidently scanned double or triple

# Flagzerotelling = True # Perform a zerotelling
# Flagstockcorrecties = True # Perform stock correctie
# Flagdayremnants = True # Perform checks in case the colli from the delivery does not fit in the shelf 
# Flagorderprediction = True # Predict when to order new products 

# #################################################Start generation of data###############################################


# for product_id in range(0, nr_products): #Loop over alll products
#     prod_init = ip.Initialize_prod(T, Shelf_length) 
#     price = prod_init["price"]
#     theft = prod_init["theft"]
#     sold = prod_init["sold"]
#     rest = prod_init["rest"]
#     max_prod_rows = prod_init["max_prod_rows"]
#     size = prod_init["size"]
#     prod_max = prod_init["prod_max"]
#     prod_min = prod_init["prod_min"]
#     Colli = prod_init["Colli"]
#     max_delivery = prod_init["max_delivery"]
#     Delivery_time = prod_init["Delivery_time"]
#     days_after_order = prod_init["days_after_order"]
#     Delivery_scheduled = prod_init["Delivery_scheduled"]
#     new_order = prod_init["new_order"]
#     delivery_amount = prod_init["delivery_amount"]
#     double_scanned = prod_init["double_scanned"]
#     extra_colli = prod_init["extra_colli"]
#     Extra_colli_stock = prod_init["Extra_colli_stock"]
#     coll_stat = prod_init["coll_stat"]
#     stock_missing = prod_init["stock_missing"]
#     zerotell = prod_init["zerotell"]
#     zero_t = prod_init["zero_t"]
#     stock = prod_init["stock"]
#     actual_stock = prod_init["actual_stock"]
#     stock_correctie = prod_init["stock_correctie"]
#     stock_cor = prod_init["stock_cor"]
#     day_rest_cor = prod_init["day_rest_cor"]
#     pred_t = prod_init["pred_t"]
#     avg_delivery_time = prod_init["avg_delivery_time"] 
#     days_till_order = None
#     Frist_order_day = True
    
#     #Loop over every timestep 
#     for t in range(1, T):

#         #Precompute actual stock
#         actual_next_stock = (
#             actual_stock[t-1]
#             - sold[t]
#             - theft[t]
#             - rest[t]
#             #+ Vault_coll_prod # In case colli was missing 1 product!
#             )

#         if actual_next_stock < 0:
#             sold[t], theft[t], rest[t] = fsg.allocate_stock(
#                 actual_stock[t-1],
#                 sold[t],
#                 theft[t],
#                 rest[t]
#             )

#             #Correct actual stock
#             actual_next_stock = (
#                 actual_stock[t-1]
#                 - sold[t]
#                 - theft[t]
#                 - rest[t]
#                 #+ Vault_coll_prod # In case colli was missing 1 product!
#                 )
        
#         #Compute next stock in system with precomputed losses
#         next_stock = (
#             stock[t-1]
#             - sold[t]
#             - theft[t]
#             - rest[t]
#             #+ Vault_coll_prod # In case colli was missing 1 product!
#             )

#         ####Manual zero-telling (System says stock the size of a Colli or more is present but reality says we have no products)####
#         # Check every 7 days, 
#         #done in the morning!
#         if t % 7 == 0 and t <= T and Flagzerotelling == True: 
#             #print("zero telling mogelijkheid t =" , t)
            
#             if actual_stock[t-1] == 0:
#                 #print("product_id =", product_id)
#                 #print("Correction t =",t)
#                 next_stock = 0
#                 zero_t[t] = 1
#                 sold[t] =0 
#                 theft[t] = 0
#                 rest[t] = 0
#                 actual_next_stock = 0
#                 #print("zero telling t =", t)
#         else:
#             zero_t[t] = 0
        
# ############################################################################################
#         if not np.isnan(time_to_order):
#             days_till_order = int(time_to_order) - 1
#             first_order_day = True
        
#             print("first flagged")
#             print(t)
#             print("time_to_order =", time_to_order)
#             print(days_till_order)
        
#         elif days_till_order is not None and days_till_order > 0:
#             days_till_order -= 1
        
#             print(t)
#             print(days_till_order)
        
#         elif days_till_order == 0:
#             #Here can order!
#             print(t)
#             print("order")
            
#             #days_after_order = 1
#             #new_order == True
        
#             # reset after ordering
#             days_till_order = None
#             first_order_day = False

            
# #####################################################################################

        
#         delivery_arrived = False
#         order_placed = False 
#         day_rest_check = False
#         # Add extra stock delivery is bound to arrive
#         if new_order == False:
#             if (Delivery_sceduled - days_after_order) == 0:
#                 next_stock += max_delivery * Colli
#                 actual_next_stock += max_delivery * Colli
#                 days_after_order = 0
#                 new_order = True
#                 delivery_arrived = True
#                 day_rest_check = True
#                 delivery_amount[t] = max_delivery * Colli


#                 if Flag_col_del_mis == True:
#                     values = np.arange(0, max_delivery + 1)
                
#                     # exponentially decreasing probabilities
#                     probs = np.exp(-values)
#                     probs = probs / probs.sum()
                
#                     Vault_coll = np.random.choice(values, p=probs)
#                 else:
#                     Vault_coll = 0
                
#                 coll_stat[t] = Vault_coll
    

#                 p_Colli_vault_prod = fsg.Colli_vault_prod(Vault_coll, max_delivery)

#                 if Flag_col_del_prod_mis:
#                     Vault_coll_prod = np.random.choice(
#                         np.arange(0, max_delivery + 1),
#                         p=p_Colli_vault_prod
#                     )
#                 else:
#                     Vault_coll_prod = 0
                    
    
#                 stock_missing[t] = np.abs(coll_stat[t]*Colli - Vault_coll_prod)
#                 # print("t =",t)
#                 # print("stock_missing[t] =", stock_missing[t])
     
#         #Make a order decision (which keeps track of the count when an order is made)
#         if delivery_arrived == False:
#             if new_order == True and stock[t-1] <= prod_min:
#                 Delivery_sceduled = Delivery_time[t]   
#                 days_after_order = 1
#                 new_order = False
#                 order_placed = True 
                
#             # elif new_order == True and days_till_order == 0:
#             #     Delivery_sceduled = Delivery_time[t]   
#             #     days_after_order = 1
#             #     new_order = False
#             #     order_placed = True


# ##############################
            
#             elif new_order == False:
#                 days_after_order += 1
        
#         # Ensure delivery_time is set 0 zero, (only non-zero when an order is placed!)
#         if not order_placed:
#             Delivery_time[t] = 0

                       
                                        
#         #Add randomized double_scanned in cost if scanned products (sold or rest)
#         if Flag_double_scanned == True:
#             if sold[t] > 0 or rest[t] > 0:
#                 double_scanned[t] = np.random.choice(
#                 [0, 1, 2],
#                 p=fsg.double_scan_prob(sold[t], rest[t])
#                 )
#                 next_stock = next_stock - double_scanned[t] #Register extra scanned product from stock
                
#             else:
#                 double_scanned[t] = 0 #Double but a well :)
                
#         else:
#             double_scanned[t] = 0
        

#         if Flag_col_del_extra == True:
#             #There is a small chance of 1 or 2 extra collies
#             extra_probs = np.array([0.995, 0.004, 0.001])  # corresponds to 0,1,2 extra collies
#             extra_colli[t] = np.random.choice([0, 1, 2], p=extra_probs)
#             max_extra = extra_colli[t]
            
#             # Calculate missing products per colli if there are extra collies comming
#             if max_extra > 0 and Flag_col_extra_del_prod_mis == True:
#                 volume = max_extra  # treat max_extra as "delivery size"
#                 scale = volume / max(volume, 1)  # avoid divide by zero
#                 missing_values = np.arange(0, 3)  # 0,1,2 missing products per extra colli
#                 missing_probs = np.exp(-scale * missing_values)
#                 missing_probs /= missing_probs.sum()  # normalize
            
#                 Vault_extra_coll_prod = np.random.choice(missing_values, p=missing_probs)
#             else:
#                 Vault_extra_coll_prod = 0
#         else:
#             extra_colli[t] = 0
#             Vault_extra_coll_prod = 0
        
            
#         Extra_colli_stock[t] = int(extra_colli[t]) * Colli - int(Vault_extra_coll_prod)
            
            
#         #Daily overload products from deliveries
#         if Flagdayremnants == True and day_rest_check == True: # Only perform a check in case you get a delivery!
#             if (actual_next_stock + Extra_colli_stock[t] - stock_missing[t]) > prod_max:
#                 next_stock = actual_next_stock + Extra_colli_stock[t] - stock_missing[t]
#                 day_rest_cor[t] = 1

        
#         # stock corrections
        

#         ###Manual stock correctie In case stock becomes negative you correct this product ones every 7 days in the evening preferably :)
#         # Check every 7 days on wensdays, (two days apart from zerotelling), 
#         #done in the evening after delivery is processed on wensdays (thus missing collies and products from deliveries from that day are also taken into account!)

#         if Flagstockcorrecties == True:
#             #print("prod = ",product_id)
#             if stock[t-1] < 0:
#                 #print("First time neg stock t =", t)
#                 #print("t =", t)
#                 stock_correctie = True
                
#             if (t+2) % 7 == 0 and t <= T and stock_correctie == True: 
#                 #print("stock correctie at t =",t)
#                 next_stock = actual_next_stock + Extra_colli_stock[t] - stock_missing[t]
#                 stock_cor[t] = 1 
#                 stock_correctie = False
                             

#         #Update stock shown in system 
#         stock.append(next_stock)
#         #Update the actual stock 
#         actual_stock[t] = actual_next_stock + Extra_colli_stock[t] - stock_missing[t]

# ########################################Stock prediction algo##############################################################    
#         if Flagorderprediction == True:
#             #print(t)
#             base_prior_pmf, states, history, draws, multi_draws, base_history, time_to_order = pmf_script.stock_prediction(
#                 t, 
#                 Idx_pred,
#                 stock,
#                 sold,
#                 delivery_amount,
#                 rng,
#                 pred_t,
#                 avg_delivery_time,
#                 window,
#                 lag, 
#                 decay,
#                 rho,
#                 sigma,
#                 base_prior_pmf,
#                 states,
#                 history,
#                 draws,
#                 multi_draws,
#                 base_history,)

#             # if time_to_order is not np.nan:
#             #     print(t)
#             #     print(time_to_order)
#             #print(multi_draws)
#             #print(t)
#             #print(time_to_order)
#             #print(base_prior_pmf)
#             #print(time_to_order)

            
    
#     ########Add all variables to a dataframe################
#     df_product = pd.DataFrame({
#         "Product": [product_id]*T,
#         "time": list(range(1, T+1)),
#         "theft": theft,
#         "Price": [price]*T,
#         "stock": stock,
#         "sold": sold,
#         "Rest": rest,
#         "prod_max": prod_max,
#         "prod_min":prod_min,
#         "max_delivery": max_delivery,
#         "stock missing (C_M+P_M_C)": stock_missing,
#         "Double scanned": double_scanned,
#         "Colli extra (EX_C+P_M_C)": Extra_colli_stock,
#         "actual_stock": actual_stock,
#         "Colli": [Colli]*T,
#         "day rest correctie": day_rest_cor, 
#         "stock correctie": stock_cor,
#         "zerotelling": zero_t,
#         "Delivery time": Delivery_time,
#         "delivery_amount": delivery_amount,
#     })

#     all_products.append(df_product)
    
#     empty_row = pd.DataFrame({
#     "Product": [None],
#     "time": [None],
#     "theft": [None],
#     "Price": [None],
#     "stock": [None],
#     "sold": [None],
#     "Rest": [None],
#     "prod_max": [None],
#     "prod_min":[None],
#     "max_delivery": [None],
#     "stock missing (C_M+P_M_C)": [None],
#     "Double scanned": [None],
#     "Colli extra (EX_C+P_M_C)": [None],
#     "actual_stock": [None],
#     "Colli": [None],
#     "day rest correctie": [None],
#     "stock correctie": [None],
#     "zerotelling": [None],
#     "Delivery time": [None],
#     "delivery_amount": [None],
#     })
#     all_products.append(empty_row)

# # Combine dataset
# df = pd.concat(all_products, ignore_index=True)

# #Save data from first product 
# one_product = df[df["Product"] == 0]

In [ ]:
# import Functions_stock_generation as fsg # Import stock generation functions
# import Initialize_product as ip # Call Variables from initialization
# from pmf_stock_pred import pmf_stock_predictions as pmf_script# Call functions from pmf prediction script



# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt


# #############################################Create data set#############################################

# #Set seed for random generations
# np.random.seed(25)
# #Set seed for pmf prediction)
# rng = np.random.default_rng(42)
            
# #Initialize dataframe and time periods
# all_products = []
# columns = ["Product", "time", "theft", "Price", "Size",
#            "stock", "sold", "Rest", "Double scanned", "Extra colli",
#            "Colli", "zerotelling"]

# empty_row = pd.DataFrame([{col: "" for col in columns}]) # create empty row for readability
# T = 365*2  # total time periods 2 years!
# Shelf_length = 1 # 1 meters
# Idx_pred = int(np.round(T/2)) #Set Idx for stock prediction 


# #initialize variables for short and long timescales pmf updating 
# window = 28 #4 weeks, trend (note this can matter a lot)
# lag = 50 #time scale for rolling window base_pmf
# decay = 0.9 #decay used for updating the pmf
# rho = 0.03 #parameter for updating the base_pmf
# sigma = 0.8 #parameter for updating the base_pmf
# base_prior_pmf = 0 #for updating the base_pmf
# states = 0 #for updating the states in base_pmf
# history = []
# draws = []
# multi_draws = []
# base_history = []
        


# #Calculations for size (needed for theft later on??)
# max_lenght, max_width, max_height = 0.25, 0.25, 0.25
# min_lenght, min_width, min_height = 0.1, 0.1, 0.1
# min_size = min_lenght * min_width * min_height 
# max_size = max_lenght * max_width * max_height 


# #Flags

# Flag_col_del_mis = True # missing collies
# Flag_col_del_prod_mis = True #Missing products within collies (normal delivery only)

# Flag_col_del_extra = True # Not registerd extra colli delivery 
# Flag_col_extra_del_prod_mis = True # Missing products within extra collies
# Flag_double_scanned = True # Products accidently scanned double or triple

# Flagzerotelling = True # Perform a zerotelling
# Flagstockcorrecties = True # Perform stock correctie
# Flagdayremnants = True # Perform checks in case the colli from the delivery does not fit in the shelf 
# Flagorderprediction = True # Predict when to order new products 

# #################################################Start generation of data###############################################


# for product_id in range(0, 1): #Loop over alll products
#     prod_init = ip.Initialize_prod(T, Shelf_length) 
#     price = prod_init["price"]
#     theft = prod_init["theft"]
#     sold = prod_init["sold"]
#     rest = prod_init["rest"]
#     max_prod_rows = prod_init["max_prod_rows"]
#     size = prod_init["size"]
#     prod_max = prod_init["prod_max"]
#     prod_min = prod_init["prod_min"]
#     Colli = prod_init["Colli"]
#     max_delivery = prod_init["max_delivery"]
#     Delivery_time = prod_init["Delivery_time"]
#     days_after_order = prod_init["days_after_order"]
#     Delivery_scheduled = prod_init["Delivery_scheduled"]
#     new_order = prod_init["new_order"]
#     delivery_amount = prod_init["delivery_amount"]
#     double_scanned = prod_init["double_scanned"]
#     extra_colli = prod_init["extra_colli"]
#     Extra_colli_stock = prod_init["Extra_colli_stock"]
#     coll_stat = prod_init["coll_stat"]
#     stock_missing = prod_init["stock_missing"]
#     zerotell = prod_init["zerotell"]
#     zero_t = prod_init["zero_t"]
#     stock = prod_init["stock"]
#     actual_stock = prod_init["actual_stock"]
#     stock_correctie = prod_init["stock_correctie"]
#     stock_cor = prod_init["stock_cor"]
#     day_rest_cor = prod_init["day_rest_cor"]
#     pred_t = prod_init["pred_t"]
#     avg_delivery_time = prod_init["avg_delivery_time"] 

    
#     #Loop over every timestep 
#     for t in range(1, T):

#         #Precompute actual stock
#         actual_next_stock = (
#             actual_stock[t-1]
#             - sold[t]
#             - theft[t]
#             - rest[t]
#             #+ Vault_coll_prod # In case colli was missing 1 product!
#             )

#         if actual_next_stock < 0:
#             sold[t], theft[t], rest[t] = fsg.allocate_stock(
#                 actual_stock[t-1],
#                 sold[t],
#                 theft[t],
#                 rest[t]
#             )

#             #Correct actual stock
#             actual_next_stock = (
#                 actual_stock[t-1]
#                 - sold[t]
#                 - theft[t]
#                 - rest[t]
#                 #+ Vault_coll_prod # In case colli was missing 1 product!
#                 )
        
#         #Compute next stock in system with precomputed losses
#         next_stock = (
#             stock[t-1]
#             - sold[t]
#             - theft[t]
#             - rest[t]
#             #+ Vault_coll_prod # In case colli was missing 1 product!
#             )

#         ####Manual zero-telling (System says stock the size of a Colli or more is present but reality says we have no products)####
#         # Check every 7 days, 
#         #done in the morning!
#         if t % 7 == 0 and t <= T and Flagzerotelling == True: 
#             #print("zero telling mogelijkheid t =" , t)
            
#             if actual_stock[t-1] == 0:
#                 #print("product_id =", product_id)
#                 #print("Correction t =",t)
#                 next_stock = 0
#                 zero_t[t] = 1
#                 sold[t] =0 
#                 theft[t] = 0
#                 rest[t] = 0
#                 actual_next_stock = 0
#                 #print("zero telling t =", t)
#         else:
#             zero_t[t] = 0
        

#         delivery_arrived = False
#         order_placed = False 
#         day_rest_check = False
#         # Add extra stock delivery is bound to arrive
#         if new_order == False:
#             if (Delivery_sceduled - days_after_order) == 0:
#                 next_stock += max_delivery * Colli
#                 actual_next_stock += max_delivery * Colli
#                 days_after_order = 0
#                 new_order = True
#                 delivery_arrived = True
#                 day_rest_check = True
#                 delivery_amount[t] = max_delivery * Colli


#                 if Flag_col_del_mis == True:
#                     values = np.arange(0, max_delivery + 1)
                
#                     # exponentially decreasing probabilities
#                     probs = np.exp(-values)
#                     probs = probs / probs.sum()
                
#                     Vault_coll = np.random.choice(values, p=probs)
#                 else:
#                     Vault_coll = 0
                
#                 coll_stat[t] = Vault_coll
    

#                 p_Colli_vault_prod = fsg.Colli_vault_prod(Vault_coll, max_delivery)

#                 if Flag_col_del_prod_mis:
#                     Vault_coll_prod = np.random.choice(
#                         np.arange(0, max_delivery + 1),
#                         p=p_Colli_vault_prod
#                     )
#                 else:
#                     Vault_coll_prod = 0
                    
    
#                 stock_missing[t] = np.abs(coll_stat[t]*Colli - Vault_coll_prod)
#                 # print("t =",t)
#                 # print("stock_missing[t] =", stock_missing[t])
     
#         #Make a order decision (which keeps track of the count when an order is made)
#         if delivery_arrived == False:
#             if new_order == True and stock[t-1] <= prod_min:
#                 Delivery_sceduled = Delivery_time[t]   
#                 days_after_order = 1
#                 new_order = False
#                 order_placed = True                    
#             elif new_order == False:
#                 days_after_order += 1
        
#         # Ensure delivery_time is set 0 zero, (only non-zero when an order is placed!)
#         if not order_placed:
#             Delivery_time[t] = 0

                       
                                        
#         #Add randomized double_scanned in cost if scanned products (sold or rest)
#         if Flag_double_scanned == True:
#             if sold[t] > 0 or rest[t] > 0:
#                 double_scanned[t] = np.random.choice(
#                 [0, 1, 2],
#                 p=fsg.double_scan_prob(sold[t], rest[t])
#                 )
#                 next_stock = next_stock - double_scanned[t] #Register extra scanned product from stock
                
#             else:
#                 double_scanned[t] = 0 #Double but a well :)
                
#         else:
#             double_scanned[t] = 0
        

#         if Flag_col_del_extra == True:
#             #There is a small chance of 1 or 2 extra collies
#             extra_probs = np.array([0.98, 0.015, 0.005])  # corresponds to 0,1,2 extra collies
#             extra_colli[t] = np.random.choice([0, 1, 2], p=extra_probs)
#             max_extra = extra_colli[t]
            
#             # Calculate missing products per colli if there are extra collies comming
#             if max_extra > 0 and Flag_col_extra_del_prod_mis == True:
#                 volume = max_extra  # treat max_extra as "delivery size"
#                 scale = volume / max(volume, 1)  # avoid divide by zero
#                 missing_values = np.arange(0, 3)  # 0,1,2 missing products per extra colli
#                 missing_probs = np.exp(-scale * missing_values)
#                 missing_probs /= missing_probs.sum()  # normalize
            
#                 Vault_extra_coll_prod = np.random.choice(missing_values, p=missing_probs)
#             else:
#                 Vault_extra_coll_prod = 0
#         else:
#             extra_colli[t] = 0
#             Vault_extra_coll_prod = 0
        
            
#         Extra_colli_stock[t] = int(extra_colli[t]) * Colli - int(Vault_extra_coll_prod)
            
            
#         #Daily overload products from deliveries
#         if Flagdayremnants == True and day_rest_check == True: # Only perform a check in case you get a delivery!
#             if (actual_next_stock + Extra_colli_stock[t] - stock_missing[t]) > prod_max:
#                 next_stock = actual_next_stock + Extra_colli_stock[t] - stock_missing[t]
#                 day_rest_cor[t] = 1

        
#         # stock corrections
        

#         ###Manual stock correctie In case stock becomes negative you correct this product ones every 7 days in the evening preferably :)
#         # Check every 7 days on wensdays, (two days apart from zerotelling), 
#         #done in the evening after delivery is processed on wensdays (thus missing collies and products from deliveries from that day are also taken into account!)

#         if Flagstockcorrecties == True:
#             #print("prod = ",product_id)
#             if stock[t-1] < 0:
#                 #print("First time neg stock t =", t)
#                 #print("t =", t)
#                 stock_correctie = True
                
#             if (t+2) % 7 == 0 and t <= T and stock_correctie == True: 
#                 #print("stock correctie at t =",t)
#                 next_stock = actual_next_stock + Extra_colli_stock[t] - stock_missing[t]
#                 stock_cor[t] = 1 
#                 stock_correctie = False
                             

#         #Update stock shown in system 
#         stock.append(next_stock)
#         #Update the actual stock 
#         actual_stock[t] = actual_next_stock + Extra_colli_stock[t] - stock_missing[t]

# ########################################Stock prediction algo##############################################################    
#         if Flagorderprediction == True:
#             #print(t)
#             base_prior_pmf, states, history, draws, multi_draws, base_history, time_to_order = pmf_script.stock_prediction(
#                 t, 
#                 Idx_pred,
#                 stock,
#                 sold,
#                 delivery_amount,
#                 rng,
#                 pred_t,
#                 avg_delivery_time,
#                 window,
#                 lag, 
#                 decay,
#                 rho,
#                 sigma,
#                 base_prior_pmf,
#                 states,
#                 history,
#                 draws,
#                 multi_draws,
#                 base_history,)
#             #print(multi_draws)
#             #print(t)
        
#     #         if t == Idx_pred: #When half of data data is generated initialize Base pmf (with weighted component)
    
#     #             #Determine epsilon 
#     #             # #s^t+1_i = s^t_i + delivery^t_i + epsilon (note here this is the approximation of epsilon!)
#     #             epsilon = np.asarray(stock, dtype=int) - np.roll(np.asarray(stock, dtype=int),-1)
#     #             epsilon, base_pmf_epsilon = pmf.update_epsilon(epsilon, stock, sold , Idx_pred)
                
#     #             #Calculate possible states in pmf
#     #             states = pmf.make_states_from_data(epsilon, tail_buffer=5)
    
#     #             #Calculate pmf
#     #             base_prior_pmf = pmf.pmf_from_samples_with_tail_prior(
#     #                 #stock_prior, #stock_prior
#     #                 epsilon,
#     #                 states=states,
#     #                 prior_strength=2.0, #Parameter
#     #                 lam=1.2 #Parameter
#     #             )
                
    
#     #             #initialize variables for short and long timescales updating
#     #             window = 28 #4 weeks, trend (note this can matter a lot)
#     #             lag = 50 #time scale for rolling window base_pmf
#     #             decay = 0.9 #decay used for updating the pmf
#     #             rho = 0.03 #parameter for updating the base_pmf
#     #             sigma = 0.8 #parameter for updating the base_pmf
                
#     #             history = []
#     #             draws = []
#     #             multi_draws = []
#     #             base_history = []
                
                
#     #         elif t > Idx_pred: #Start predicting
    
#     #             #Update epsilon
#     #             # #s^t+1_i = s^t_i + delivery^t_i + epsilon (note here this is the approximation of epsilon!)
#     #             epsilon = np.asarray(stock, dtype=int) - np.roll(np.asarray(stock, dtype=int),-1)
#     #             epsilon, base_pmf_epsilon = pmf.update_epsilon(epsilon, stock, sold , Idx_pred)
    
#     #             #select previous data
#     #             recent_data = epsilon[-window:] #stock[Idx_pred+recent_start : Idx_pred+t]
    
#     #             #Update pmf
#     #             pmf_t = pmf.predictive_pmf_from_base_and_recent(
#     #                 base_prior_pmf=base_prior_pmf,
#     #                 recent_data=recent_data,
#     #                 states=states,
#     #                 decay=decay)
    
                
#     #             #Keep track of base_pmf and updated pmf_t
#     #             history.append(pmf_t)
#     #             base_history.append(base_prior_pmf.copy())
                
#     # ##############################Make random draws finish this part!!###################################
                
#     #             if delivery_amount[t] > 0:
#     #                 draws = rng.choice(states, size=200, p=pmf_t) #200 is high but certainly enough!
#     #                 multi_draws.append(draws) #Save the draws
    
#     #                 # start value
#     #                 start_val = stock[t]
                
#     #                 # cumulative subtraction (prepend 0 so first value stays start_val)
#     #                 pred_after_deli = start_val - np.concatenate(([0], np.cumsum(draws))) #Make new array and substract cumulative sum over elements
#     #                 pred_t[t].append(pred_after_deli) #Save for plotting 
    
    
#     #                 #Make a prediction at which time in the future to order
#     #                 j_pred = 0
#     #                 #time_to_order
#     #                 while j_pred <= len(pred_after_deli):
#     #                     pred_stock = pred_after_deli[j_pred]
                    
#     #                     if pred_stock <= 0:
                            
#     #                         time_to_order = j_pred - avg_delivery_time -1 #Calulate when to order (-1 is for extra certaintly)
#     #                         print("time_to_order =", time_to_order)
#     #                         # print(" j_pred=", j_pred)
#     #                         # print("avg_delivery_time =", avg_delivery_time)
#     #                         # print("pred deli arival=", pred_after_deli[time_to_order + avg_delivery_time])
#     #                         # print("pred_after_deli=", pred_after_deli)
                            
#     #                         # print("prod_min = ", prod_min)
#     #                         # print("pred_after_deli[j_pred] =", pred_after_deli[j_pred])
#     #                         # print("pred_after_deli[j_pred] =", pred_after_deli[j_pred])
#     #                         # print(stock[t])
#     #                         # print(pred_after_deli)
    
              
#     #                         break
#     #                     j_pred += 1
                    
#     #                     if j_pred > 200:
#     #                         break
                    
        
#     #             #update baseline only with lagged datapoint
#     #             lag_index = t - lag
#     #             if not np.isnan(base_pmf_epsilon[lag_index]):
#     #                 base_prior_pmf = pmf.update_base_prior_lagged(
#     #                     base_prior_pmf=base_prior_pmf,
#     #                     obs=base_pmf_epsilon[lag_index], 
#     #                     states=states,
#     #                     rho=rho,
#     #                     sigma=sigma)
        
    
    
#     # #############################################################################################################

    
    
#     ########Add all variables to a dataframe################
#     df_product = pd.DataFrame({
#         "Product": [product_id]*T,
#         "time": list(range(1, T+1)),
#         "theft": theft,
#         "Price": [price]*T,
#         "stock": stock,
#         "sold": sold,
#         "Rest": rest,
#         "prod_max": prod_max,
#         "prod_min":prod_min,
#         "max_delivery": max_delivery,
#         "stock missing (C_M+P_M_C)": stock_missing,
#         "Double scanned": double_scanned,
#         "Colli extra (EX_C+P_M_C)": Extra_colli_stock,
#         "actual_stock": actual_stock,
#         "Colli": [Colli]*T,
#         "day rest correctie": day_rest_cor, 
#         "stock correctie": stock_cor,
#         "zerotelling": zero_t,
#         "Delivery time": Delivery_time,
#         "delivery_amount": delivery_amount,
#     })

#     all_products.append(df_product)
    
#     empty_row = pd.DataFrame({
#     "Product": [None],
#     "time": [None],
#     "theft": [None],
#     "Price": [None],
#     "stock": [None],
#     "sold": [None],
#     "Rest": [None],
#     "prod_max": [None],
#     "prod_min":[None],
#     "max_delivery": [None],
#     "stock missing (C_M+P_M_C)": [None],
#     "Double scanned": [None],
#     "Colli extra (EX_C+P_M_C)": [None],
#     "actual_stock": [None],
#     "Colli": [None],
#     "day rest correctie": [None],
#     "stock correctie": [None],
#     "zerotelling": [None],
#     "Delivery time": [None],
#     "delivery_amount": [None],
#     })
#     all_products.append(empty_row)

# # Combine dataset
# df = pd.concat(all_products, ignore_index=True)

# #Save data from first product 
# one_product = df[df["Product"] == 0]

In [ ]:
# import Functions_stock_generation as fsg # Import stock generation functions
# import Initialize_product as ip # Call Variables from initialization


# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt


# #############################################Create data set#############################################

# #Set seed for random generations
# np.random.seed(25)

# #Initialize dataframe and time periods
# all_products = []
# columns = ["Product", "time", "theft", "Price", "Size",
#            "stock", "sold", "Rest", "Double scanned", "Extra colli",
#            "Colli", "zerotelling"]

# empty_row = pd.DataFrame([{col: "" for col in columns}]) # create empty row for readability
# T = 365*2  # total time periods 2 years!
# Shelf_length = 1 # 1 meters
# Idx_pred = int(np.round(T/2)) #Set Idx for stock prediction 


# #Calculations for size (needed for theft later on??)
# max_lenght, max_width, max_height = 0.25, 0.25, 0.25
# min_lenght, min_width, min_height = 0.1, 0.1, 0.1
# min_size = min_lenght * min_width * min_height 
# max_size = max_lenght * max_width * max_height 


# #Flags

# Flag_col_del_mis = True # missing collies
# Flag_col_del_prod_mis = True #Missing products within collies (normal delivery only)

# Flag_col_del_extra = True # Not registerd extra colli delivery 
# Flag_col_extra_del_prod_mis = True # Missing products within extra collies
# Flag_double_scanned = True # Products accidently scanned double or triple

# Flagzerotelling = True # Perform a zerotelling
# Flagstockcorrecties = True # Perform stock correctie
# Flagdayremnants = True # Perform checks in case the colli from the delivery does not fit in the shelf 

# #################################################Start generation of data###############################################


# for product_id in range(0, 50): #Loop over alll products
#     prod_init = ip.Initialize_prod(T, Shelf_length) 
#     price = prod_init["price"]
#     theft = prod_init["theft"]
#     sold = prod_init["sold"]
#     rest = prod_init["rest"]
#     max_prod_rows = prod_init["max_prod_rows"]
#     size = prod_init["size"]
#     prod_max = prod_init["prod_max"]
#     prod_min = prod_init["prod_min"]
#     Colli = prod_init["Colli"]
#     max_delivery = prod_init["max_delivery"]
#     Delivery_time = prod_init["Delivery_time"]
#     days_after_order = prod_init["days_after_order"]
#     Delivery_scheduled = prod_init["Delivery_scheduled"]
#     new_order = prod_init["new_order"]
#     delivery_amount = prod_init["delivery_amount"]
#     double_scanned = prod_init["double_scanned"]
#     extra_colli = prod_init["extra_colli"]
#     Extra_colli_stock = prod_init["Extra_colli_stock"]
#     coll_stat = prod_init["coll_stat"]
#     stock_missing = prod_init["stock_missing"]
#     zerotell = prod_init["zerotell"]
#     zero_t = prod_init["zero_t"]
#     stock = prod_init["stock"]
#     actual_stock = prod_init["actual_stock"]
#     stock_correctie = prod_init["stock_correctie"]
#     stock_cor = prod_init["stock_cor"]
#     day_rest_cor = prod_init["day_rest_cor"]
    

    
#     #Loop over every timestep 
#     for t in range(1, T):

#         #Precompute actual stock
#         actual_next_stock = (
#             actual_stock[t-1]
#             - sold[t]
#             - theft[t]
#             - rest[t]
#             #+ Vault_coll_prod # In case colli was missing 1 product!
#             )

#         if actual_next_stock < 0:
#             sold[t], theft[t], rest[t] = fsg.allocate_stock(
#                 actual_stock[t-1],
#                 sold[t],
#                 theft[t],
#                 rest[t]
#             )

#             #Correct actual stock
#             actual_next_stock = (
#                 actual_stock[t-1]
#                 - sold[t]
#                 - theft[t]
#                 - rest[t]
#                 #+ Vault_coll_prod # In case colli was missing 1 product!
#                 )
        
#         #Compute next stock in system with precomputed losses
#         next_stock = (
#             stock[t-1]
#             - sold[t]
#             - theft[t]
#             - rest[t]
#             #+ Vault_coll_prod # In case colli was missing 1 product!
#             )

#         ####Manual zero-telling (System says stock the size of a Colli or more is present but reality says we have no products)####
#         # Check every 7 days, 
#         #done in the morning!
#         if t % 7 == 0 and t <= T and Flagzerotelling == True: 
#             #print("zero telling mogelijkheid t =" , t)
            
#             if actual_stock[t-1] == 0:
#                 #print("product_id =", product_id)
#                 #print("Correction t =",t)
#                 next_stock = 0
#                 zero_t[t] = 1
#                 sold[t] =0 
#                 theft[t] = 0
#                 rest[t] = 0
#                 actual_next_stock = 0
#                 #print("zero telling t =", t)
#         else:
#             zero_t[t] = 0
        

#         delivery_arrived = False
#         order_placed = False 
#         day_rest_check = False
#         # Add extra stock delivery is bound to arrive
#         if new_order == False:
#             if (Delivery_sceduled - days_after_order) == 0:
#                 next_stock += max_delivery * Colli
#                 actual_next_stock += max_delivery * Colli
#                 days_after_order = 0
#                 new_order = True
#                 delivery_arrived = True
#                 day_rest_check = True
#                 delivery_amount[t] = max_delivery * Colli


#                 if Flag_col_del_mis == True:
#                     values = np.arange(0, max_delivery + 1)
                
#                     # exponentially decreasing probabilities
#                     probs = np.exp(-values)
#                     probs = probs / probs.sum()
                
#                     Vault_coll = np.random.choice(values, p=probs)
#                 else:
#                     Vault_coll = 0
                
#                 coll_stat[t] = Vault_coll
    

#                 p_Colli_vault_prod = fsg.Colli_vault_prod(Vault_coll, max_delivery)

#                 if Flag_col_del_prod_mis:
#                     Vault_coll_prod = np.random.choice(
#                         np.arange(0, max_delivery + 1),
#                         p=p_Colli_vault_prod
#                     )
#                 else:
#                     Vault_coll_prod = 0
                    
    
#                 stock_missing[t] = np.abs(coll_stat[t]*Colli - Vault_coll_prod)
#                 # print("t =",t)
#                 # print("stock_missing[t] =", stock_missing[t])
     
#         #Make a order decision (which keeps track of the count when an order is made)
#         if delivery_arrived == False:
#             if new_order == True and stock[t-1] <= prod_min:
#                 Delivery_sceduled = Delivery_time[t]   
#                 days_after_order = 1
#                 new_order = False
#                 order_placed = True                    
#             elif new_order == False:
#                 days_after_order += 1
        
#         # Ensure delivery_time is set 0 zero, (only non-zero when an order is placed!)
#         if not order_placed:
#             Delivery_time[t] = 0

                       
                                        
#         #Add randomized double_scanned in cost if scanned products (sold or rest)
#         if Flag_double_scanned == True:
#             if sold[t] > 0 or rest[t] > 0:
#                 double_scanned[t] = np.random.choice(
#                 [0, 1, 2],
#                 p=fsg.double_scan_prob(sold[t], rest[t])
#                 )
#                 next_stock = next_stock - double_scanned[t] #Register extra scanned product from stock
                
#             else:
#                 double_scanned[t] = 0 #Double but a well :)
                
#         else:
#             double_scanned[t] = 0
        

#         if Flag_col_del_extra == True:
#             #There is a small chance of 1 or 2 extra collies
#             extra_probs = np.array([0.98, 0.015, 0.005])  # corresponds to 0,1,2 extra collies
#             extra_colli[t] = np.random.choice([0, 1, 2], p=extra_probs)
#             max_extra = extra_colli[t]
            
#             # Calculate missing products per colli if there are extra collies comming
#             if max_extra > 0 and Flag_col_extra_del_prod_mis == True:
#                 volume = max_extra  # treat max_extra as "delivery size"
#                 scale = volume / max(volume, 1)  # avoid divide by zero
#                 missing_values = np.arange(0, 3)  # 0,1,2 missing products per extra colli
#                 missing_probs = np.exp(-scale * missing_values)
#                 missing_probs /= missing_probs.sum()  # normalize
            
#                 Vault_extra_coll_prod = np.random.choice(missing_values, p=missing_probs)
#             else:
#                 Vault_extra_coll_prod = 0
#         else:
#             extra_colli[t] = 0
#             Vault_extra_coll_prod = 0
        
            
#         Extra_colli_stock[t] = int(extra_colli[t]) * Colli - int(Vault_extra_coll_prod)
            
            
#         #Daily overload products from deliveries
#         if Flagdayremnants == True and day_rest_check == True: # Only perform a check in case you get a delivery!
#             if (actual_next_stock + Extra_colli_stock[t] - stock_missing[t]) > prod_max:
#                 next_stock = actual_next_stock + Extra_colli_stock[t] - stock_missing[t]
#                 day_rest_cor[t] = 1

        
#         # stock corrections
        

#         ###Manual stock correctie In case stock becomes negative you correct this product ones every 7 days in the evening preferably :)
#         # Check every 7 days on wensdays, (two days apart from zerotelling), 
#         #done in the evening after delivery is processed on wensdays (thus missing collies and products from deliveries from that day are also taken into account!)

#         if Flagstockcorrecties == True:
#             #print("prod = ",product_id)
#             if stock[t-1] < 0:
#                 #print("First time neg stock t =", t)
#                 #print("t =", t)
#                 stock_correctie = True
                
#             if (t+2) % 7 == 0 and t <= T and stock_correctie == True: 
#                 #print("stock correctie at t =",t)
#                 next_stock = actual_next_stock + Extra_colli_stock[t] - stock_missing[t]
#                 stock_cor[t] = 1 
#                 stock_correctie = False
                             

#         #Update stock shown in system 
#         stock.append(next_stock)
#         #Update the actual stock 
#         actual_stock[t] = actual_next_stock + Extra_colli_stock[t] - stock_missing[t]

        

    
#     ########Add all variables to a dataframe################
#     df_product = pd.DataFrame({
#         "Product": [product_id]*T,
#         "time": list(range(1, T+1)),
#         "theft": theft,
#         "Price": [price]*T,
#         "stock": stock,
#         "sold": sold,
#         "Rest": rest,
#         "prod_max": prod_max,
#         "prod_min":prod_min,
#         "max_delivery": max_delivery,
#         "stock missing (C_M+P_M_C)": stock_missing,
#         "Double scanned": double_scanned,
#         "Colli extra (EX_C+P_M_C)": Extra_colli_stock,
#         "actual_stock": actual_stock,
#         "Colli": [Colli]*T,
#         "day rest correctie": day_rest_cor, 
#         "stock correctie": stock_cor,
#         "zerotelling": zero_t,
#         "Delivery time": Delivery_time,
#         "delivery_amount": delivery_amount,
#     })

#     all_products.append(df_product)
    
#     empty_row = pd.DataFrame({
#     "Product": [None],
#     "time": [None],
#     "theft": [None],
#     "Price": [None],
#     "stock": [None],
#     "sold": [None],
#     "Rest": [None],
#     "prod_max": [None],
#     "prod_min":[None],
#     "max_delivery": [None],
#     "stock missing (C_M+P_M_C)": [None],
#     "Double scanned": [None],
#     "Colli extra (EX_C+P_M_C)": [None],
#     "actual_stock": [None],
#     "Colli": [None],
#     "day rest correctie": [None],
#     "stock correctie": [None],
#     "zerotelling": [None],
#     "Delivery time": [None],
#     "delivery_amount": [None],
#     })
#     all_products.append(empty_row)

# # Combine dataset
# df = pd.concat(all_products, ignore_index=True)

# #Save data from first product 
# one_product = df[df["Product"] == 2]